# LeetCode 4068 — Maximize Meeting Earnings with Idle Gaps (Hard)

Bài toán LeetCode (Weekly Contest 521, câu 4):

```python
class Solution:
    def maxEarnings(self, meetings: List[List[int]]) -> int:
        ...
```

Đây là một bài **chọn khoảng thời gian không chồng nhau** (interval scheduling) có thêm một "gia vị" lạ:
thời gian **rảnh** giữa hai cuộc họp kề nhau cũng mang lại tiền. Chính gia vị này làm bài khó hơn
bài kinh điển *Maximum Profit in Job Scheduling* — nhưng nếu biết **biến đổi công thức một chút**,
ta lại đưa được nó về đúng khuôn mẫu quen thuộc: **sắp xếp + quy hoạch động (DP) + tìm kiếm nhị phân**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc hiểu chính xác đề: khoảng **nửa mở** `[start, end)`, thế nào là "chồng nhau", thời gian rảnh được tính ở đâu.
2. Tự tính tay thu nhập của một lịch họp và tự liệt kê mọi lựa chọn cho một ví dụ nhỏ.
3. Hiểu cách đặt **trạng thái DP** dạng "lịch tốt nhất *kết thúc tại* cuộc họp `i`".
4. Biết mẹo **tách biến**: `max_j (A_i + B_j) = A_i + max_j B_j` — mấu chốt để từ `O(n²)` xuống `O(n log n)`.
5. Dùng thành thạo `sorted(..., key=...)`, `bisect_right` / `bisect_left`, mảng **prefix max** và **hai con trỏ (sweep)**.
6. Chứng minh được vì sao thuật toán đúng và phân tích độ phức tạp của từng cách.
7. Nhận diện họ bài "weighted interval scheduling" và chuyển kỹ thuật sang các biến thể.

---

## Các cách giải sẽ học (từ dễ đến tối ưu)

| Cách | Ý tưởng | Thời gian |
|---|---|---|
| **Cách 1** | Brute force: thử mọi tập con cuộc họp | `O(2^n · n²)` |
| **Cách 2** | DP `f[i]` = lịch tốt nhất kết thúc tại `i`, xét mọi `j` trước đó | `O(n²)` |
| **Cách 3** | DP + tách biến + **prefix max** + **binary search** (`bisect_right`) | `O(n log n)` |
| **Cách 4** | DP + tách biến + **sweep hai con trỏ** (duyệt theo `start`, không cần bisect) | `O(n log n)` |

# 1. Đề bài (dịch sát nghĩa)

Cho mảng 2 chiều `meetings`, trong đó `meetings[i] = [start_i, end_i, revenue_i]` là một cuộc họp
**bắt đầu** lúc `start_i`, **kết thúc** lúc `end_i`, mang lại **doanh thu** (revenue) `revenue_i`.

- Mọi cuộc họp dùng **khoảng nửa mở** `[start, end)`: thời điểm `start` thuộc cuộc họp, thời điểm `end` thì **không**.
  Vì vậy hai cuộc họp **chỉ chạm nhau ở đầu mút** (ví dụ `[1, 3)` và `[3, 5)`) thì **không chồng nhau**.
- Bạn được chọn **một tập con khác rỗng** các cuộc họp sao cho **không có hai cuộc họp được chọn nào chồng nhau**.
  Bạn nhận doanh thu của mỗi cuộc họp được chọn.
- Xếp các cuộc họp được chọn theo `start` tăng dần. Với **mỗi cặp cuộc họp kề nhau** trong thứ tự đó,
  bạn nhận thêm **1 đơn vị doanh thu cho mỗi đơn vị thời gian rảnh** (idle time) giữa chúng.
  Thời gian rảnh = `start` của cuộc họp sau − `end` của cuộc họp trước.
- **Không** có tiền rảnh trước cuộc họp sớm nhất, cũng **không** có tiền rảnh sau cuộc họp muộn nhất.
  Nếu chỉ chọn 1 cuộc họp thì không có tiền rảnh nào.

**Trả về tổng thu nhập lớn nhất** có thể đạt được.

Skeleton cần viết:

```python
class Solution:
    def maxEarnings(self, meetings: List[List[int]]) -> int:
```

## Ví dụ trong đề

**Ví dụ 1**

```text
Input:  meetings = [[2,5,4],[6,8,3]]
Output: 8
```

Chọn cả hai: doanh thu `4 + 3 = 7`; cuộc họp đầu kết thúc lúc 5, cuộc họp sau bắt đầu lúc 6 → rảnh `6 − 5 = 1`.
Tổng `7 + 1 = 8`.

**Ví dụ 2**

```text
Input:  meetings = [[3,5,4],[4,7,8],[8,10,3]]
Output: 12
```

Chọn cuộc họp index 1 và 2: doanh thu `8 + 3 = 11`; họp `[4,7)` rồi `[8,10)` → rảnh `8 − 7 = 1`. Tổng `12`.

**Ví dụ 3**

```text
Input:  meetings = [[1,2,2],[4,5,2],[7,9,3]]
Output: 11
```

Chọn cả ba: doanh thu `2 + 2 + 3 = 7`; rảnh `4 − 2 = 2` và `7 − 5 = 2`. Tổng `7 + 2 + 2 = 11`.

---

## Ràng buộc (constraints)

- `1 <= meetings.length <= 10^5`
- `meetings[i] = [start_i, end_i, revenue_i]`
- `0 <= start_i < end_i <= 10^9`
- `1 <= revenue_i <= 10^9`

## Ràng buộc nói gì về thuật toán?

| Ràng buộc | Hệ quả |
|---|---|
| `n` tới `10^5` | `O(n²) ≈ 10^10` phép tính → quá chậm. Cần khoảng `O(n log n)`. |
| thời gian tới `10^9` | **Không** thể tạo mảng theo từng đơn vị thời gian (10^9 ô). Phải làm việc trên **n cuộc họp đã sắp xếp** + tìm kiếm nhị phân. |
| `revenue >= 1` | Mọi cuộc họp đều "có lời" về doanh thu — điều này sẽ cho ta một nhận xét rất hữu ích ở phần đột phá. |
| tổng có thể ≈ `10^5 · 10^9 + 10^9 ≈ 10^14` | Vượt `int` 32-bit. Python không lo tràn số, nhưng C++/Java phải dùng `long long` / `long`. |

In [1]:
examples = [
    ([[2, 5, 4], [6, 8, 3]], 8),
    ([[3, 5, 4], [4, 7, 8], [8, 10, 3]], 12),
    ([[1, 2, 2], [4, 5, 2], [7, 9, 3]], 11),
]

def draw_timeline(meetings):
    """Vẽ mỗi cuộc họp thành một hàng: '#' ở ô [t, t+1) nằm trong cuộc họp."""
    T = max(e for _, e, _ in meetings)
    print("            " + "".join(str(t % 10) for t in range(T)))
    for idx, (s, e, r) in enumerate(meetings):
        row = "".join("#" if s <= t < e else "." for t in range(T))
        print(f"[{idx}] r={r:<3}   {row}")

for meetings, expected in examples:
    print("meetings =", meetings, "| expected =", expected)
    draw_timeline(meetings)
    print()

meetings = [[2, 5, 4], [6, 8, 3]] | expected = 8
            01234567
[0] r=4     ..###...
[1] r=3     ......##

meetings = [[3, 5, 4], [4, 7, 8], [8, 10, 3]] | expected = 12
            0123456789
[0] r=4     ...##.....
[1] r=8     ....###...
[2] r=3     ........##

meetings = [[1, 2, 2], [4, 5, 2], [7, 9, 3]] | expected = 11
            012345678
[0] r=2     .#.......
[1] r=2     ....#....
[2] r=3     .......##



Đọc hình: cột số ở trên là thời điểm `t`; ký tự `#` ở cột `t` nghĩa là ô thời gian `[t, t+1)` nằm trong cuộc họp.

Ở ví dụ 2, hàng `[0]` (`[3,5)`) và hàng `[1]` (`[4,7)`) cùng có `#` ở cột 4 → **chồng nhau**, không thể chọn cả hai.
Đó là lý do đáp án bỏ cuộc họp `[0]`.

# 2. Hiểu đề bằng tay trước khi nghĩ thuật toán

## 2.1 Thu nhập của một lịch họp

Ta viết một hàm nhỏ `earnings_of(selected)`: nhận một danh sách cuộc họp **đã chọn**, trả về thu nhập.

Các bước làm tay:

1. Xếp các cuộc họp đã chọn theo `start`.
2. Cộng tất cả `revenue`.
3. Với mỗi cặp kề nhau `(trước, sau)`: cộng `gap = sau.start − trước.end`.
4. Nếu có `gap < 0` thì hai cuộc họp kề nhau chồng lên nhau → lịch **không hợp lệ** (trả về `None`).

Ví dụ 3, chọn cả ba:

| cặp kề nhau | trước.end | sau.start | gap |
|---|---:|---:|---:|
| `[1,2)` → `[4,5)` | 2 | 4 | 2 |
| `[4,5)` → `[7,9)` | 5 | 7 | 2 |

Doanh thu `2 + 2 + 3 = 7`, tiền rảnh `2 + 2 = 4` → tổng `11`.

In [2]:
def earnings_of(selected):
    """Thu nhập của một lịch họp đã chọn; None nếu có 2 cuộc họp kề nhau chồng lên nhau."""
    order = sorted(selected, key=lambda m: m[0])   # xếp theo start
    total = sum(r for _, _, r in order)            # tổng doanh thu
    for k in range(len(order) - 1):
        gap = order[k + 1][0] - order[k][1]        # start sau - end trước
        if gap < 0:
            return None                            # chồng nhau -> không hợp lệ
        total += gap                               # tiền rảnh
    return total

print(earnings_of([[1, 2, 2], [4, 5, 2], [7, 9, 3]]))   # 11
print(earnings_of([[2, 5, 4], [6, 8, 3]]))              # 8
print(earnings_of([[3, 5, 4], [4, 7, 8]]))              # None (chồng nhau)
print(earnings_of([[4, 7, 8]]))                         # 8 (chỉ 1 cuộc họp: không có tiền rảnh)

11
8
None
8


## 2.2 Liệt kê mọi lựa chọn của ví dụ 2

Gọi `A = [3,5,4]`, `B = [4,7,8]`, `C = [8,10,3]`. Có `2^3 − 1 = 7` tập con khác rỗng:

| Tập chọn | Hợp lệ? | Doanh thu | Tiền rảnh | Tổng |
|---|---|---:|---:|---:|
| {A} | có | 4 | 0 | 4 |
| {B} | có | 8 | 0 | 8 |
| {C} | có | 3 | 0 | 3 |
| {A, B} | **không** (`[3,5)` và `[4,7)` cùng chứa `[4,5)`) | – | – | – |
| {A, C} | có | 7 | `8 − 5 = 3` | 10 |
| {B, C} | có | 11 | `8 − 7 = 1` | **12** |
| {A, B, C} | **không** | – | – | – |

Đáp án = 12, khớp đề. Cell dưới kiểm chứng bảng này bằng code.

In [3]:
from itertools import combinations

names = {"A": [3, 5, 4], "B": [4, 7, 8], "C": [8, 10, 3]}
for size in range(1, 4):
    for combo in combinations("ABC", size):
        chosen = [names[c] for c in combo]
        print("{" + ", ".join(combo) + "}", "->", earnings_of(chosen))

{A} -> 4
{B} -> 8
{C} -> 3
{A, B} -> None
{A, C} -> 10
{B, C} -> 12
{A, B, C} -> None


## 2.3 Một ví dụ "bẫy": chọn nhiều hơn chưa chắc tốt hơn

Xét `meetings = [[0,1,1],[1,10,1],[10,11,1]]` (ba cuộc họp nối đuôi nhau, chạm mút).

- Chọn cả ba: doanh thu `3`, tiền rảnh `0 + 0 = 0` → tổng `3`.
- Chỉ chọn cuộc họp đầu và cuối: doanh thu `2`, tiền rảnh `10 − 1 = 9` → tổng `11`.

Cuộc họp giữa dài 9 đơn vị nhưng chỉ đem lại 1 đồng. Nếu **bỏ** nó, 9 đơn vị đó trở thành thời gian rảnh và
được trả 9 đồng! Vậy bài này **không** phải "chọn càng nhiều càng tốt". Ta cần một phương pháp xét có hệ thống.

In [4]:
trap = [[0, 1, 1], [1, 10, 1], [10, 11, 1]]
print("Chọn cả ba          :", earnings_of(trap))
print("Bỏ cuộc họp ở giữa  :", earnings_of([trap[0], trap[2]]))
print("Chỉ cuộc họp giữa   :", earnings_of([trap[1]]))

Chọn cả ba          : 3
Bỏ cuộc họp ở giữa  : 11
Chỉ cuộc họp giữa   : 1


# 3. Nền tảng cần biết

Trước khi giải, ta học (hoặc ôn) từng công cụ sẽ dùng. Mỗi mục: **nó là gì → ví dụ nhỏ → code chạy thử → liên quan gì tới bài**.

| Mục | Công cụ | Dùng ở cách nào |
|---|---|---|
| 3.1 | Khoảng nửa mở `[s, e)` và kiểm tra chồng nhau | mọi cách |
| 3.2 | `sorted(..., key=lambda ...)` và unpacking `for s, e, r in ...` | mọi cách |
| 3.3 | `itertools.combinations` — sinh tập con | Cách 1 |
| 3.4 | Quy hoạch động (DP) dạng "kết thúc tại i" | Cách 2, 3, 4 |
| 3.5 | Tách biến: `max_j (A + B_j) = A + max_j B_j` | Cách 3, 4 |
| 3.6 | Mảng prefix max và giá trị canh `float("-inf")` | Cách 3 |
| 3.7 | Tìm kiếm nhị phân `bisect_left` / `bisect_right` | Cách 3 |
| 3.8 | Hai con trỏ (sweep) trên hai thứ tự sắp xếp | Cách 4 |
| 3.9 | Số nguyên lớn trong Python và tràn số ở ngôn ngữ khác | LeetCode |

## 3.1 Khoảng nửa mở `[s, e)` — khi nào hai cuộc họp chồng nhau?

`[s, e)` chứa mọi thời điểm `t` với `s <= t < e`. Ví dụ `[1, 3)` chứa `1, 1.5, 2, 2.9...` nhưng **không chứa 3**.

Hai khoảng `a = [a_s, a_e)` và `b = [b_s, b_e)` **chồng nhau** khi có phần chung khác rỗng, tức là:

```python
a_s < b_e and b_s < a_e
```

Đọc thành lời: "a bắt đầu trước khi b kết thúc **và** b bắt đầu trước khi a kết thúc".

| a | b | chồng nhau? | lý do |
|---|---|---|---|
| `[1,3)` | `[3,5)` | không | chỉ chạm ở 3, mà 3 không thuộc `[1,3)` |
| `[1,4)` | `[3,5)` | có | cùng chứa `[3,4)` |
| `[0,10)` | `[2,3)` | có | b nằm trong a |
| `[2,6)` | `[4,6)` | có | **cùng end** luôn chồng nhau (cùng chứa đoạn ngay trước 6) |

**Liên quan tới bài:** điều kiện "cuộc họp `j` đặt được **trước** cuộc họp `i`" chính là `end_j <= start_i`
(dấu `<=`, vì chạm mút vẫn hợp lệ). Dấu `<=` này sẽ quyết định ta dùng `bisect_right` chứ không phải `bisect_left`.

In [5]:
def overlaps(a, b):
    """a, b có dạng [start, end, revenue]; True nếu hai khoảng nửa mở chồng nhau."""
    return a[0] < b[1] and b[0] < a[1]

print(overlaps([1, 3, 0], [3, 5, 0]))    # False: chỉ chạm mút
print(overlaps([1, 4, 0], [3, 5, 0]))    # True
print(overlaps([0, 10, 0], [2, 3, 0]))   # True: lồng nhau
print(overlaps([2, 6, 0], [4, 6, 0]))    # True: cùng end

False
True
True
True


## 3.2 Sắp xếp với `key` và unpacking

`sorted(lst, key=f)` trả về **danh sách mới** đã sắp theo giá trị `f(x)` của từng phần tử (danh sách gốc không đổi).
`lambda m: m[1]` là một hàm không tên: nhận `m`, trả về `m[1]` (tức `end`).

```python
sorted(meetings, key=lambda m: m[1])   # sắp theo end tăng dần
```

Python sắp xếp **ổn định** (stable): hai phần tử có cùng key giữ nguyên thứ tự ban đầu.

**Unpacking**: `for s, e, r in ms:` tách mỗi `[start, end, revenue]` thành 3 biến. Kết hợp `enumerate` để có cả chỉ số:
`for i, (s, e, r) in enumerate(ms):`.

**Liên quan tới bài:** Cách 2, 3 xử lý các cuộc họp theo thứ tự `end` tăng dần; Cách 4 duyệt theo `start` tăng dần.

In [6]:
meetings = [[3, 5, 4], [4, 7, 8], [8, 10, 3], [1, 5, 9]]
by_end = sorted(meetings, key=lambda m: m[1])
print("gốc     :", meetings)            # không bị thay đổi
print("theo end:", by_end)              # [3,5,4] đứng trước [1,5,9] vì ổn định

for i, (s, e, r) in enumerate(by_end):
    print(f"i={i}: start={s}, end={e}, revenue={r}")

gốc     : [[3, 5, 4], [4, 7, 8], [8, 10, 3], [1, 5, 9]]
theo end: [[3, 5, 4], [1, 5, 9], [4, 7, 8], [8, 10, 3]]
i=0: start=3, end=5, revenue=4
i=1: start=1, end=5, revenue=9
i=2: start=4, end=7, revenue=8
i=3: start=8, end=10, revenue=3


## 3.3 `itertools.combinations` — sinh mọi tập con

`combinations(lst, k)` sinh mọi cách chọn `k` phần tử (không lặp, giữ thứ tự gốc). Cho `k` chạy từ `1` tới `n`
ta được **mọi tập con khác rỗng** — tổng cộng `2^n − 1` tập.

Ví dụ với `["A", "B", "C"]`: `k=1` → A, B, C; `k=2` → AB, AC, BC; `k=3` → ABC. Tổng `7 = 2^3 − 1`.

**Liên quan tới bài:** Cách 1 (brute force) thử **mọi** tập con — chắc chắn đúng nhưng số tập con tăng gấp đôi
mỗi khi thêm một cuộc họp.

In [7]:
from itertools import combinations

items = ["A", "B", "C"]
all_subsets = []
for k in range(1, len(items) + 1):
    for combo in combinations(items, k):
        all_subsets.append("".join(combo))
print(all_subsets, "->", len(all_subsets), "tập con")

for n in [3, 10, 20, 30]:
    print(f"n = {n:>2}: 2^n - 1 = {2 ** n - 1:,} tập con")

['A', 'B', 'C', 'AB', 'AC', 'BC', 'ABC'] -> 7 tập con
n =  3: 2^n - 1 = 7 tập con
n = 10: 2^n - 1 = 1,023 tập con
n = 20: 2^n - 1 = 1,048,575 tập con
n = 30: 2^n - 1 = 1,073,741,823 tập con


## 3.4 Quy hoạch động (DP) dạng "kết thúc tại i"

**DP** (Dynamic Programming) = giải bài lớn bằng cách **lưu lại đáp án của các bài con** rồi ghép lại, thay vì tính lại nhiều lần.
Khi làm DP ta luôn trả lời 4 câu:

1. **Trạng thái**: `f[i]` nghĩa là gì?
2. **Chuyển trạng thái**: tính `f[i]` từ các `f[j]` nhỏ hơn thế nào?
3. **Thứ tự tính**: tính `f` theo thứ tự nào để khi cần `f[j]` thì nó đã có?
4. **Đáp án**: lấy từ đâu trong bảng `f`?

Ví dụ khởi động (không phải bài chính): *tổng lớn nhất của một dãy con tăng dần* trong `nums = [3, 1, 4, 2, 5]`.

1. `f[i]` = tổng lớn nhất của dãy con tăng **kết thúc đúng tại** `nums[i]`.
2. `f[i] = nums[i] + max(f[j] với j < i và nums[j] < nums[i])` (nếu không có `j` nào thì chỉ là `nums[i]`).
3. Tính `i` từ trái sang phải.
4. Đáp án = `max(f)` (dãy tốt nhất phải kết thúc ở *đâu đó*).

| i | nums[i] | các j hợp lệ | f[i] |
|---:|---:|---|---:|
| 0 | 3 | – | 3 |
| 1 | 1 | – | 1 |
| 2 | 4 | 0, 1 | 4 + max(3, 1) = 7 |
| 3 | 2 | 1 | 2 + 1 = 3 |
| 4 | 5 | 0, 1, 2, 3 | 5 + max(3, 1, 7, 3) = 12 |

**Liên quan tới bài:** bài của ta có cùng hình dạng: "một chuỗi các cuộc họp nối tiếp nhau", mỗi cuộc họp được nối
sau những cuộc họp *tương thích* với nó. Ta sẽ đặt `f[i]` = thu nhập tốt nhất của lịch **kết thúc tại** cuộc họp `i`.

In [8]:
nums = [3, 1, 4, 2, 5]
f = [0] * len(nums)
for i in range(len(nums)):
    f[i] = nums[i]                                   # chỉ một mình nums[i]
    for j in range(i):
        if nums[j] < nums[i]:
            f[i] = max(f[i], f[j] + nums[i])         # nối sau dãy kết thúc tại j
print("f   =", f)        # [3, 1, 7, 3, 12]
print("đáp án =", max(f))

f   = [3, 1, 7, 3, 12]
đáp án = 12


## 3.5 Mẹo tách biến: `max_j (A + B_j) = A + max_j B_j`

Nếu một biểu thức là **tổng của phần chỉ phụ thuộc `i`** và **phần chỉ phụ thuộc `j`**, thì khi lấy max theo `j`,
phần của `i` là hằng số và có thể kéo ra ngoài.

Ví dụ: `A = 10`, `B = [3, -2, 7]`.

- Cách thẳng: `max(10 + 3, 10 − 2, 10 + 7) = max(13, 8, 17) = 17`.
- Cách tách: `10 + max(3, −2, 7) = 10 + 7 = 17`.

Điều quý giá: `max B` có thể **tính sẵn một lần** rồi dùng lại cho rất nhiều `i` khác nhau.

**Liên quan tới bài:** công thức nối cuộc họp `j` trước `i` là `f[j] + revenue_i + (start_i − end_j)`.
Nhóm lại: `(revenue_i + start_i) + (f[j] − end_j)` → phần của `i` và phần của `j` tách rời. Đây là chìa khoá của bài!

In [9]:
A = 10
B = [3, -2, 7]
print("max(A + b)  =", max(A + b for b in B))
print("A + max(B)  =", A + max(B))

# Với nhiều giá trị A khác nhau, chỉ cần tính max(B) đúng một lần
best_B = max(B)
for A in [10, 0, -5]:
    assert max(A + b for b in B) == A + best_B
    print(f"A = {A:>3}: đáp án = {A + best_B}")

max(A + b)  = 17
A + max(B)  = 17
A =  10: đáp án = 17
A =   0: đáp án = 7
A =  -5: đáp án = 2


## 3.6 Mảng prefix max và giá trị canh `float("-inf")`

**Prefix max** (max tiền tố): `pre[k] = max(arr[0], arr[1], ..., arr[k-1])` — max của `k` phần tử **đầu tiên**.

Quy ước `pre[0]` = max của 0 phần tử = "không có gì". Ta dùng `float("-inf")` (âm vô cùng) làm **giá trị canh** (sentinel):
nó nhỏ hơn mọi số, nên `max(-inf, x) = x`.

Công thức cập nhật: `pre[k + 1] = max(pre[k], arr[k])` — mỗi bước `O(1)`.

Ví dụ `arr = [-1, 3, 2, 5, 0]`:

| k | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| pre[k] | -inf | -1 | 3 | 3 | 5 | 5 |

**Liên quan tới bài:** ta sẽ lưu `pre_max[k] = max(f[j] − end_j)` trên `k` cuộc họp đầu tiên (đã sắp theo `end`).
Chú ý: `pre[0] = -inf` nghĩa là "không có cuộc họp nào đứng trước được" — **khác hẳn** với `0`!

In [10]:
arr = [-1, 3, 2, 5, 0]
NEG = float("-inf")
pre = [NEG] * (len(arr) + 1)
for k in range(len(arr)):
    pre[k + 1] = max(pre[k], arr[k])
print("pre =", pre)
print("max của 3 phần tử đầu:", pre[3])
print("max(-inf, -1000000) =", max(NEG, -1_000_000))

pre = [-inf, -1, 3, 3, 5, 5]
max của 3 phần tử đầu: 3
max(-inf, -1000000) = -1000000


## 3.7 Tìm kiếm nhị phân với `bisect`

Module `bisect` làm việc trên **danh sách đã sắp xếp tăng dần**, mỗi lần gọi chỉ tốn `O(log n)`.

- `bisect_left(a, x)`: vị trí đầu tiên mà `a[pos] >= x` = **số phần tử `< x`**.
- `bisect_right(a, x)`: vị trí đầu tiên mà `a[pos] > x` = **số phần tử `<= x`**.

Hai hàm chỉ khác nhau khi `x` **có mặt** trong danh sách. Ví dụ `a = [2, 5, 5, 7, 9, 10]`:

| x | bisect_left | bisect_right | ý nghĩa của bisect_right |
|---:|---:|---:|---|
| 1 | 0 | 0 | không phần tử nào `<= 1` |
| 5 | 1 | 3 | có 3 phần tử `<= 5`: 2, 5, 5 |
| 6 | 3 | 3 | có 3 phần tử `<= 6` |
| 9 | 4 | 5 | có 5 phần tử `<= 9` |

**Liên quan tới bài:** nếu `ends` là danh sách `end` đã sắp tăng, thì `p = bisect_right(ends, start_i)` = số cuộc họp có
`end <= start_i` — chính là các cuộc họp **được phép** đứng trước `i` (kể cả trường hợp chạm mút `end == start_i`).

In [11]:
from bisect import bisect_left, bisect_right

a = [2, 5, 5, 7, 9, 10]
for x in [1, 5, 6, 9]:
    print(f"x = {x}: bisect_left = {bisect_left(a, x)}, bisect_right = {bisect_right(a, x)}")

# Kiểm chứng ý nghĩa "đếm số phần tử <= x" bằng cách đếm thủ công
for x in range(0, 12):
    assert bisect_right(a, x) == sum(1 for v in a if v <= x)
    assert bisect_left(a, x) == sum(1 for v in a if v < x)
print("bisect_right = số phần tử <= x ; bisect_left = số phần tử < x  -> đúng")

x = 1: bisect_left = 0, bisect_right = 0
x = 5: bisect_left = 1, bisect_right = 3
x = 6: bisect_left = 3, bisect_right = 3
x = 9: bisect_left = 4, bisect_right = 5
bisect_right = số phần tử <= x ; bisect_left = số phần tử < x  -> đúng


## 3.8 Hai con trỏ (sweep) trên hai thứ tự sắp xếp

Bài toán nhỏ: có danh sách `starts` (đã sắp tăng) và `ends` (đã sắp tăng). Với **mỗi** `start`, đếm số `end <= start`.

- Cách bisect: mỗi `start` gọi `bisect_right(ends, start)` → `O(log n)` mỗi lần.
- Cách **hai con trỏ**: vì `start` tăng dần, tập `{end <= start}` chỉ **lớn thêm**, không bao giờ nhỏ đi.
  Giữ một con trỏ `ptr` trên `ends`; với mỗi `start` mới, chỉ việc đẩy `ptr` tiến lên khi `ends[ptr] <= start`.
  Tổng số lần đẩy `ptr` cả quá trình ≤ `n` → tổng `O(n)`.

Đây gọi là **sweep line** (quét đường thẳng): tưởng tượng một vạch thời gian chạy từ trái sang phải,
"gom" dần những cuộc họp đã kết thúc.

**Liên quan tới bài:** Cách 4 duyệt các cuộc họp theo `start` tăng dần và gom dần các cuộc họp đã kết thúc
vào một biến `best` duy nhất — không cần mảng `pre_max`, không cần bisect.

In [12]:
from bisect import bisect_right

starts = [0, 1, 2, 3, 6, 9]
ends = [2, 5, 5, 7, 9, 10]

ptr = 0
for s in starts:
    while ptr < len(ends) and ends[ptr] <= s:
        ptr += 1                     # gom thêm một cuộc họp đã kết thúc
    assert ptr == bisect_right(ends, s)
    print(f"start = {s}: có {ptr} end <= start   (bisect_right cũng cho {bisect_right(ends, s)})")

start = 0: có 0 end <= start   (bisect_right cũng cho 0)
start = 1: có 0 end <= start   (bisect_right cũng cho 0)
start = 2: có 1 end <= start   (bisect_right cũng cho 1)
start = 3: có 1 end <= start   (bisect_right cũng cho 1)
start = 6: có 3 end <= start   (bisect_right cũng cho 3)
start = 9: có 5 end <= start   (bisect_right cũng cho 5)


## 3.9 Số nguyên lớn: Python vs C++/Java

Python có `int` **không giới hạn** độ lớn, nên tổng `10^14` vẫn chính xác. Nhưng `int` 32-bit của C++/Java chỉ chứa
tối đa `2^31 − 1 ≈ 2.1 · 10^9` — chỉ cần **hai** cuộc họp doanh thu `10^9` là đã tràn.

Khi dịch lời giải sang ngôn ngữ khác, dùng `long long` (C++) / `long` (Java) cho `f`, `pre_max`, `ans`.

In [13]:
INT32_MAX = 2 ** 31 - 1
INT64_MAX = 2 ** 63 - 1
worst = 10 ** 5 * 10 ** 9 + 10 ** 9         # cận trên thô của đáp án
print("INT32_MAX =", f"{INT32_MAX:,}")
print("đáp án tối đa ~", f"{worst:,}")
print("vượt int32?", worst > INT32_MAX, "| vừa int64?", worst <= INT64_MAX)

INT32_MAX = 2,147,483,647
đáp án tối đa ~ 100,001,000,000,000
vượt int32? True | vừa int64? True


# 4. Cách 1 — Brute force: thử mọi tập con

## Ý tưởng

Làm đúng như đề nói, không thông minh gì cả:

1. Sinh **mọi tập con khác rỗng** của `meetings` (`combinations` với `k = 1..n`).
2. Bỏ những tập có **hai cuộc họp bất kỳ** chồng nhau (kiểm tra mọi cặp bằng `overlaps`).
3. Với tập hợp lệ, tính thu nhập bằng `earnings_of` và giữ giá trị lớn nhất.

Vì nó bám sát từng câu chữ của đề, ta dùng nó làm **"trọng tài"** để kiểm tra các cách nhanh hơn ở sau.

In [14]:
from itertools import combinations
from typing import List

def max_earnings_bruteforce(meetings: List[List[int]]) -> int:
    best = None
    n = len(meetings)
    for k in range(1, n + 1):
        for chosen in combinations(meetings, k):
            # (1) không được có cặp nào chồng nhau
            if any(overlaps(a, b) for a, b in combinations(chosen, 2)):
                continue
            # (2) tính thu nhập của lịch hợp lệ
            value = earnings_of(chosen)
            if best is None or value > best:
                best = value
    return best

for meetings, expected in examples:
    result = max_earnings_bruteforce(meetings)
    print(meetings, "->", result, "| expected =", expected)
    assert result == expected
print("Brute force đúng trên cả 3 ví dụ của đề.")

[[2, 5, 4], [6, 8, 3]] -> 8 | expected = 8
[[3, 5, 4], [4, 7, 8], [8, 10, 3]] -> 12 | expected = 12
[[1, 2, 2], [4, 5, 2], [7, 9, 3]] -> 11 | expected = 11
Brute force đúng trên cả 3 ví dụ của đề.


## Tại sao `earnings_of` chỉ cần kiểm tra các cặp **kề nhau**?

Brute force kiểm tra **mọi cặp** cho chắc. Nhưng thật ra, sau khi xếp theo `start`, nếu mọi cặp kề nhau đều có `gap >= 0` thì:

```text
end_1 <= start_2 < end_2 <= start_3 < end_3 <= ...
```

→ cuộc họp 1 kết thúc trước khi cuộc họp 3, 4, ... bắt đầu, nên **mọi** cặp đều không chồng nhau.
Nhận xét nhỏ này cho ta một hệ quả quan trọng: **trong một lịch hợp lệ, thứ tự theo `start` cũng chính là thứ tự theo `end`**.

## Bao nhiêu việc?

- Số tập con: `2^n − 1`.
- Mỗi tập kiểm tra tối đa `n(n−1)/2` cặp.

→ `O(2^n · n²)`. Với `n = 20` đã là hàng trăm triệu phép tính; với `n = 10^5` thì không tưởng.
Cell dưới đếm số tập con mà brute force phải xét.

In [15]:
import math, time

for n in [10, 20, 40, 100_000]:
    if n <= 40:
        print(f"n = {n:>6}: {2 ** n - 1:,} tập con")
    else:
        digits = int(n * math.log10(2)) + 1          # số chữ số của 2^n
        print(f"n = {n:>6}: 2^{n} - 1 tập con (một số có {digits:,} chữ số!)")

# Đo thử thời gian brute force trên n = 12 (chỉ 4095 tập con)
demo = [[i, i + 2, i + 1] for i in range(12)]
t0 = time.perf_counter()
max_earnings_bruteforce(demo)
print(f"n = 12 mất khoảng {time.perf_counter() - t0:.3f} giây — mỗi +1 cuộc họp, thời gian ~x2.")

n =     10: 1,023 tập con
n =     20: 1,048,575 tập con
n =     40: 1,099,511,627,775 tập con
n = 100000: 2^100000 - 1 tập con (một số có 30,103 chữ số!)
n = 12 mất khoảng 0.004 giây — mỗi +1 cuộc họp, thời gian ~x2.


# 5. Bước đột phá 1 — Mỗi lịch đều có một "cuộc họp cuối cùng"

Thay vì nhìn cả tập con, hãy nhìn **cuộc họp kết thúc muộn nhất** của lịch. Gọi nó là `i`.

- Mọi lịch hợp lệ (khác rỗng) đều có đúng một cuộc họp cuối như vậy.
- Nếu lịch có nhiều hơn một cuộc họp, gọi `j` là cuộc họp **ngay trước** `i`. Khi đó `end_j <= start_i`, và tiền rảnh mới phát sinh là `start_i − end_j`.
- Bỏ `i` đi, phần còn lại là một lịch hợp lệ **kết thúc tại `j`**.

Vậy nếu đặt

> `f[i]` = thu nhập lớn nhất của một lịch hợp lệ mà **cuộc họp cuối cùng là `i`**,

thì:

```text
f[i] = revenue_i                                            (chỉ họp i một mình)
f[i] = max over j có end_j <= start_i của:
           f[j] + revenue_i + (start_i - end_j)             (nối i sau lịch kết thúc tại j)
```

và **đáp án = max(f[i])** với mọi `i`.

Vì sao dùng `f[j]` (giá trị **tốt nhất**) cho phần trước là đủ? Vì phần thu nhập thêm `revenue_i + (start_i − end_j)`
chỉ phụ thuộc vào `j`, **không** phụ thuộc các cuộc họp trước `j`. Muốn tổng lớn nhất thì phần trước cũng phải lớn nhất.

**Thứ tự tính:** nếu sắp các cuộc họp theo `end` tăng dần, thì mọi `j` hợp lệ (`end_j <= start_i < end_i`) đều có
`end_j < end_i` → `j` đứng **trước** `i`. Vậy duyệt theo `end` tăng dần là khi cần `f[j]` nó đã được tính.

Ví dụ 3 (đã sắp theo end): `[1,2,2]`, `[4,5,2]`, `[7,9,3]`.

| i | cuộc họp | j hợp lệ | f[i] |
|---:|---|---|---:|
| 0 | `[1,2,2]` | – | 2 |
| 1 | `[4,5,2]` | 0 | `2 + 2 + (4 − 2) = 6` |
| 2 | `[7,9,3]` | 0, 1 | `max(2 + 3 + (7 − 2), 6 + 3 + (7 − 5)) = max(10, 11) = 11` |

In [16]:
ms = sorted([[1, 2, 2], [4, 5, 2], [7, 9, 3]], key=lambda m: m[1])
f = []
for i, (s, e, r) in enumerate(ms):
    options = [("một mình", r)]
    for j in range(i):
        sj, ej, rj = ms[j]
        if ej <= s:
            options.append((f"nối sau j={j}", f[j] + r + (s - ej)))
    best = max(v for _, v in options)
    f.append(best)
    print(f"i={i} {ms[i]}: lựa chọn = {options} -> f[{i}] = {best}")
print("đáp án = max(f) =", max(f))

i=0 [1, 2, 2]: lựa chọn = [('một mình', 2)] -> f[0] = 2
i=1 [4, 5, 2]: lựa chọn = [('một mình', 2), ('nối sau j=0', 6)] -> f[1] = 6
i=2 [7, 9, 3]: lựa chọn = [('một mình', 3), ('nối sau j=0', 10), ('nối sau j=1', 11)] -> f[2] = 11
đáp án = max(f) = 11


# 6. Bước đột phá 2 — Nối vào cuối luôn có lời, chèn vào giữa thì chưa chắc

Hai thí nghiệm nhỏ giúp hiểu "tính kinh tế" của bài.

**(a) Nối thêm một cuộc họp `k` vào cuối lịch** (sau cuộc họp cuối `last`, với `end_last <= start_k`):
thu nhập tăng `revenue_k + (start_k − end_last)`. Vì `revenue_k >= 1` và khoảng rảnh `>= 0`, phần tăng **luôn dương**.

→ Hệ quả: trong công thức `f[i]`, nếu tồn tại **ít nhất một** `j` hợp lệ thì phương án "nối" luôn tốt hơn "một mình":
`f[j] + revenue_i + (start_i − end_j) >= 1 + revenue_i > revenue_i`. Vậy ta **không cần** so sánh với "một mình"
khi đã có `j` hợp lệ.

**(b) Chèn một cuộc họp `k` vào giữa hai cuộc họp `a`, `b`** (với `end_a <= start_k`, `end_k <= start_b`):
khoảng rảnh cũ `start_b − end_a` bị cắt mất đúng **độ dài** `d_k = end_k − start_k` của cuộc họp. Thu nhập thay đổi
`revenue_k − d_k` — có thể **âm**! Đó là lý do "chọn càng nhiều càng tốt" sai (ví dụ bẫy ở mục 2.3).

Một cách nhìn gọn cho cả hai thí nghiệm (cộng dồn các khoảng rảnh — "telescoping"):

```text
tổng thu nhập = (end_cuối − start_đầu) + Σ (revenue_k − d_k)
```

Tức là: "được trả 1 đồng mỗi đơn vị thời gian từ lúc bắt đầu tới lúc kết thúc lịch, cộng thêm phần chênh `revenue − độ dài` của từng cuộc họp".

In [17]:
def span_formula(selected):
    order = sorted(selected, key=lambda m: m[0])
    span = order[-1][1] - order[0][0]
    return span + sum(r - (e - s) for s, e, r in order)

base = [[0, 2, 3], [9, 10, 1]]
print("lịch gốc:", base, "-> thu nhập", earnings_of(base))

for extra in ([3, 5, 4], [6, 9, 2], [10, 12, 1]):
    s, e, r = extra
    new = base + [extra]
    delta = earnings_of(new) - earnings_of(base)
    print(f"thêm {extra}: revenue - độ dài = {r - (e - s):>2} | thu nhập mới = {earnings_of(new)} | chênh lệch = {delta:+d}")

# Công thức telescoping khớp với cách tính trực tiếp
for sel in ([[1, 2, 2], [4, 5, 2], [7, 9, 3]], [[2, 5, 4], [6, 8, 3]], base):
    assert span_formula(sel) == earnings_of(sel)
print("span_formula == earnings_of trên các lịch thử -> OK")

lịch gốc: [[0, 2, 3], [9, 10, 1]] -> thu nhập 11
thêm [3, 5, 4]: revenue - độ dài =  2 | thu nhập mới = 13 | chênh lệch = +2
thêm [6, 9, 2]: revenue - độ dài = -1 | thu nhập mới = 10 | chênh lệch = -1
thêm [10, 12, 1]: revenue - độ dài = -1 | thu nhập mới = 12 | chênh lệch = +1
span_formula == earnings_of trên các lịch thử -> OK


Đọc kết quả:

- Chèn `[3,5,4]` vào giữa: `revenue − độ dài = 4 − 2 = 2` → thu nhập tăng đúng `+2`.
- Chèn `[6,9,2]` vào giữa: `2 − 3 = −1` → thu nhập **giảm** `1`. Không nên chọn!
- Nối `[10,12,1]` vào **cuối**: dù `revenue − độ dài = 1 − 2 = −1`, thu nhập vẫn tăng `revenue + gap = 1 + 0 = +1`.
  Theo công thức telescoping: phần `end_cuối − start_đầu` dài thêm 2, bù cho `−1` → tổng `+1`. Nối vào cuối luôn có lời;
  chỉ chèn vào **giữa** mới có thể lỗ.

Bây giờ ta đã đủ ý để viết Cách 2.

# 7. Cách 2 — DP `O(n²)`: xét mọi cuộc họp đứng trước

## Các bước

1. Sắp `meetings` theo `end` tăng dần → `ms`.
2. Với mỗi `i` theo thứ tự đó:
   - khởi tạo `f[i] = revenue_i` (một mình);
   - với mọi `j < i` mà `end_j <= start_i`: `f[i] = max(f[i], f[j] + revenue_i + (start_i − end_j))`.
3. Trả về `max(f)`.

Ta tách thành hàm `dp_quadratic_table` (trả về cả bảng `f` để quan sát) và hàm `max_earnings_dp_quadratic` (chỉ trả đáp án).

In [18]:
def dp_quadratic_table(meetings: List[List[int]]):
    ms = sorted(meetings, key=lambda m: m[1])      # sắp theo end
    n = len(ms)
    f = [0] * n
    for i, (s, e, r) in enumerate(ms):
        f[i] = r                                   # một mình
        for j in range(i):                         # mọi cuộc họp kết thúc sớm hơn
            sj, ej, rj = ms[j]
            if ej <= s:                            # j đặt được trước i
                f[i] = max(f[i], f[j] + r + (s - ej))
    return ms, f

def max_earnings_dp_quadratic(meetings: List[List[int]]) -> int:
    _, f = dp_quadratic_table(meetings)
    return max(f)

for meetings, expected in examples:
    ms, f = dp_quadratic_table(meetings)
    print("ms =", ms)
    print("f  =", f, "-> max =", max(f), "| expected =", expected)
    assert max(f) == expected

ms = [[2, 5, 4], [6, 8, 3]]
f  = [4, 8] -> max = 8 | expected = 8
ms = [[3, 5, 4], [4, 7, 8], [8, 10, 3]]
f  = [4, 8, 12] -> max = 12 | expected = 12
ms = [[1, 2, 2], [4, 5, 2], [7, 9, 3]]
f  = [2, 6, 11] -> max = 11 | expected = 11


## Độ phức tạp của Cách 2 và nút thắt

- Sắp xếp: `O(n log n)`.
- Hai vòng lặp lồng nhau: `0 + 1 + ... + (n−1) = n(n−1)/2` lần xét cặp → `O(n²)`.
- Bộ nhớ: `O(n)` cho `ms` và `f`.

Với `n = 10^5`: khoảng `5 · 10^9` lần xét cặp → quá chậm (Python chạy cỡ `10^7` phép/giây).

**Công việc nào đang lặp lại?** Với mỗi `i`, vòng `for j` tính lại `max` trên gần như cùng một tập `j` — tập đó chỉ
**dài thêm** khi `i` tăng. Nếu với mọi `i` ta đều cần "max của cùng một đại lượng trên một tiền tố", thì có thể nhớ sẵn!
Vấn đề: đại lượng `f[j] + revenue_i + (start_i − end_j)` lại **chứa `i`**... Cell dưới đo thời gian để thấy tốc độ tăng bậc hai.

In [19]:
import random, time

rng = random.Random(0)
def random_meetings(n, T=10 ** 6, rng=rng):
    out = []
    for _ in range(n):
        s = rng.randint(0, T - 1)
        e = rng.randint(s + 1, min(T, s + T // 50))
        out.append([s, e, rng.randint(1, 10 ** 4)])
    return out

for n in [500, 1000, 2000]:
    data = random_meetings(n)
    t0 = time.perf_counter()
    max_earnings_dp_quadratic(data)
    print(f"n = {n:>4}: {time.perf_counter() - t0:.3f} giây")
print("n tăng x2 -> thời gian tăng khoảng x4 (dấu hiệu của O(n^2)).")

n =  500: 0.018 giây
n = 1000: 0.051 giây


n = 2000: 0.191 giây
n tăng x2 -> thời gian tăng khoảng x4 (dấu hiệu của O(n^2)).


# 8. Bước đột phá 3 — Tách biến: phần của `i` và phần của `j`

Viết lại phương án nối:

```text
f[j] + revenue_i + (start_i - end_j)
  = (revenue_i + start_i)  +  (f[j] - end_j)
     \___ chỉ phụ thuộc i __/   \_ chỉ phụ thuộc j _/
```

Theo mẹo ở mục 3.5:

```text
f[i] = revenue_i + start_i + max( f[j] - end_j  với mọi j có end_j <= start_i )
```

(nếu không có `j` nào hợp lệ thì `f[i] = revenue_i`; nếu có thì theo Bước đột phá 2, phương án nối chắc chắn tốt hơn một mình).

Ý nghĩa của `key[j] = f[j] − end_j`: "giá trị của lịch kết thúc tại `j`, nhưng **trừ trước** thời điểm kết thúc".
Khi cuộc họp `i` nối vào, ta **cộng lại** `start_i` — hiệu `start_i − end_j` chính là tiền rảnh. Nhờ vậy, việc chọn `j`
tốt nhất **không còn phụ thuộc vào `i`**: `j` nào có `key[j]` lớn nhất thì tốt nhất cho **mọi** `i`.

Còn một câu hỏi: tập `{j : end_j <= start_i}` trông thế nào? Khi `ms` đã sắp theo `end`, các `end_j <= start_i` nằm liền
nhau ở **đầu** danh sách → là một **tiền tố** (prefix) độ dài `p = bisect_right(ends, start_i)`.

→ Kết hợp: `max(key[j])` trên tiền tố độ dài `p` = `pre_max[p]` (mục 3.6). Mỗi `i` chỉ còn `O(log n)` cho bisect + `O(1)` tra bảng!

In [20]:
from bisect import bisect_right

# Thí nghiệm trên một ví dụ 6 cuộc họp (sẽ dùng lại ở phần mô phỏng)
TRACE_MEETINGS = [[1, 7, 6], [6, 9, 2], [0, 2, 3], [9, 10, 1], [2, 5, 1], [3, 5, 4]]
ms, f = dp_quadratic_table(TRACE_MEETINGS)
ends = [e for _, e, _ in ms]
key = [f[j] - ms[j][1] for j in range(len(ms))]
print("ms   =", ms)
print("ends =", ends)
print("f    =", f)
print("key  = f - end =", key)
print()
for i, (s, e, r) in enumerate(ms):
    valid = [ends[j] <= s for j in range(len(ms))]
    p = bisect_right(ends, s)
    print(f"i={i}, start={s}: hợp lệ = {['x' if v else '.' for v in valid]}  -> tiền tố dài p = {p}")

ms   = [[0, 2, 3], [2, 5, 1], [3, 5, 4], [1, 7, 6], [6, 9, 2], [9, 10, 1]]
ends = [2, 5, 5, 7, 9, 10]
f    = [3, 4, 8, 6, 11, 13]
key  = f - end = [1, -1, 3, -1, 2, 3]

i=0, start=0: hợp lệ = ['.', '.', '.', '.', '.', '.']  -> tiền tố dài p = 0
i=1, start=2: hợp lệ = ['x', '.', '.', '.', '.', '.']  -> tiền tố dài p = 1
i=2, start=3: hợp lệ = ['x', '.', '.', '.', '.', '.']  -> tiền tố dài p = 1
i=3, start=1: hợp lệ = ['.', '.', '.', '.', '.', '.']  -> tiền tố dài p = 0
i=4, start=6: hợp lệ = ['x', 'x', 'x', '.', '.', '.']  -> tiền tố dài p = 3
i=5, start=9: hợp lệ = ['x', 'x', 'x', 'x', 'x', '.']  -> tiền tố dài p = 5


Quan sát:

- Dấu `x` (cuộc họp `j` đặt được trước `i`) **luôn dồn về bên trái** → đúng là một tiền tố, độ dài bằng `bisect_right(ends, start)`.
- Ở `i = 1` (`[2,5,1]`, start = 2): `ends[0] = 2 <= 2` → được tính (chạm mút). Nếu dùng `bisect_left` sẽ bỏ sót.
- Ở `i = 2` (`[3,5,4]`): cuộc họp `[2,5,1]` có cùng `end = 5` **không** được tính (cùng end thì chồng nhau). Tiền tố dài 1.
- `p` luôn `<= i`: các cuộc họp có `end <= start_i < end_i` đều đứng trước `i`, nên `pre_max[p]` đã được tính xong khi ta tới `i`.

Cell dưới kiểm chứng tách biến: với `i = 5` (`[9,10,1]`), cách "thẳng" và cách "tách" cho cùng giá trị, và `j` tốt nhất là `j` có `key` lớn nhất.

In [21]:
i = 5
s, e, r = ms[i]
p = bisect_right(ends, s)
print(f"i = {i}, cuộc họp {ms[i]}, các j hợp lệ: 0..{p - 1}")
for j in range(p):
    direct = f[j] + r + (s - ms[j][1])
    print(f"  j={j} {ms[j]}: f[j] + r + (s - end_j) = {direct:>2} | key[j] = f[j] - end_j = {key[j]:>2}")
direct_best = max(f[j] + r + (s - ms[j][1]) for j in range(p))
split_best = r + s + max(key[:p])
print("cách thẳng:", direct_best, "| cách tách: r + s + max(key) =", r, "+", s, "+", max(key[:p]), "=", split_best)
assert direct_best == split_best == f[i]

i = 5, cuộc họp [9, 10, 1], các j hợp lệ: 0..4
  j=0 [0, 2, 3]: f[j] + r + (s - end_j) = 11 | key[j] = f[j] - end_j =  1
  j=1 [2, 5, 1]: f[j] + r + (s - end_j) =  9 | key[j] = f[j] - end_j = -1
  j=2 [3, 5, 4]: f[j] + r + (s - end_j) = 13 | key[j] = f[j] - end_j =  3
  j=3 [1, 7, 6]: f[j] + r + (s - end_j) =  9 | key[j] = f[j] - end_j = -1
  j=4 [6, 9, 2]: f[j] + r + (s - end_j) = 12 | key[j] = f[j] - end_j =  2
cách thẳng: 13 | cách tách: r + s + max(key) = 1 + 9 + 3 = 13


# 9. Cách 3 — DP + tách biến + prefix max + `bisect_right` (`O(n log n)`)

## Ghép các mảnh lại

```text
ms      = meetings sắp theo end tăng dần
ends    = [end của từng cuộc họp trong ms]
pre_max = [-inf] * (n + 1)          # pre_max[k] = max(f[j] - end_j) với j < k
ans     = 0

với i, (s, e, r) trong ms:
    p = bisect_right(ends, s)       # số cuộc họp có end <= s  (đều có chỉ số < i)
    nếu p == 0:  f = r              # không ai đứng trước được -> họp một mình
    ngược lại:   f = r + s + pre_max[p]
    ans = max(ans, f)
    pre_max[i + 1] = max(pre_max[i], f - e)
trả về ans
```

Ba điều cần để ý:

1. **`bisect_right`** (không phải `bisect_left`) vì `end_j == start_i` vẫn hợp lệ.
2. **`pre_max[0] = -inf`** chứ không phải `0`: "không có ai đứng trước" khác với "có một cuộc họp ảo thu nhập 0 kết thúc ở thời điểm 0"
   (cái sau sẽ cộng nhầm tiền rảnh **trước** cuộc họp đầu tiên — đề cấm).
3. Ta dùng `sorted(...)` để **không làm thay đổi** danh sách đầu vào của người gọi.

In [22]:
from bisect import bisect_right
from typing import List

def max_earnings_bisect(meetings: List[List[int]]) -> int:
    ms = sorted(meetings, key=lambda m: m[1])        # 1) sắp theo end
    ends = [e for _, e, _ in ms]
    NEG = float("-inf")
    pre_max = [NEG] * (len(ms) + 1)                  # pre_max[k] = max(f[j] - end_j), j < k
    ans = 0
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)                    # 2) số cuộc họp kết thúc <= s
        if p == 0:
            f = r                                    #    không ai đứng trước -> một mình
        else:
            f = r + s + pre_max[p]                   # 3) nối sau j tốt nhất
        ans = max(ans, f)
        pre_max[i + 1] = max(pre_max[i], f - e)      # 4) cập nhật prefix max
    return ans

for meetings, expected in examples:
    result = max_earnings_bisect(meetings)
    print(meetings, "->", result, "| expected =", expected)
    assert result == expected
print("Cách 3 đúng trên 3 ví dụ.")

[[2, 5, 4], [6, 8, 3]] -> 8 | expected = 8
[[3, 5, 4], [4, 7, 8], [8, 10, 3]] -> 12 | expected = 12
[[1, 2, 2], [4, 5, 2], [7, 9, 3]] -> 11 | expected = 11
Cách 3 đúng trên 3 ví dụ.


## Độ phức tạp của Cách 3

Gọi `n` = số cuộc họp.

- Sắp xếp: `O(n log n)`.
- Vòng lặp `n` lần, mỗi lần một `bisect_right` `O(log n)` + vài phép `O(1)` → `O(n log n)`.
- Bộ nhớ: `ms`, `ends`, `pre_max` → `O(n)`.

**Tổng: Time `O(n log n)`, Space `O(n)`** — thoải mái với `n = 10^5`. Cell dưới đo thử trên `10^5` cuộc họp ngẫu nhiên
với toạ độ tới `10^9` (thời gian cụ thể tuỳ máy, thường dưới 1 giây).

In [23]:
import random, time

rng_big = random.Random(2024)
big = []
for _ in range(10 ** 5):
    s = rng_big.randint(0, 10 ** 9 - 1)
    e = rng_big.randint(s + 1, min(10 ** 9, s + 10 ** 7))
    big.append([s, e, rng_big.randint(1, 10 ** 9)])

t0 = time.perf_counter()
ans_big = max_earnings_bisect(big)
print(f"n = 100000: đáp án = {ans_big:,} | thời gian ~ {time.perf_counter() - t0:.2f} giây")
print("đáp án vượt int32?", ans_big > 2 ** 31 - 1)

n = 100000: đáp án = 1,593,117,825,737 | thời gian ~ 0.06 giây
đáp án vượt int32? True


# 10. Cách 4 — Sweep hai con trỏ: duyệt theo `start`, gom dần các cuộc họp đã kết thúc

Cách 3 cần mảng `pre_max` vì mỗi `i` hỏi một tiền tố **khác nhau**. Có cách nào chỉ giữ **một** biến `best`?

## Quan sát

Nếu `j` đặt được trước `i` thì `start_j < end_j <= start_i` → `start_j < start_i` (nhỏ hơn **chặt**).

Vậy nếu xử lý các cuộc họp theo **`start` tăng dần**:

- mọi `j` hợp lệ cho `i` đều đã được xử lý trước `i` → `f[j]` đã có;
- khi `start_i` tăng, tập `{j : end_j <= start_i}` chỉ **lớn thêm** (mục 3.8).

## Thuật toán

1. `by_start` = các cuộc họp sắp theo `start`; `order_end` = chỉ số của chúng sắp theo `end`.
2. Duyệt `i` theo `by_start`. Trước khi tính `f[i]`, đẩy con trỏ `ptr` trên `order_end`: mọi `j` có `end_j <= start_i`
   được "gom" vào `best = max(best, f[j] − end_j)`.
3. `f[i] = revenue_i` nếu chưa gom được ai (`best is None`), ngược lại `f[i] = revenue_i + start_i + best`.
4. Đáp án = `max(f)`.

Ý nghĩa của `best` (invariant): **ngay trước khi tính `f[i]`, `best` = max của `f[j] − end_j` trên mọi cuộc họp đã kết thúc trước hoặc đúng lúc `start_i`**.

In [24]:
def max_earnings_sweep(meetings: List[List[int]]) -> int:
    by_start = sorted(meetings, key=lambda m: m[0])               # duyệt theo start
    n = len(by_start)
    order_end = sorted(range(n), key=lambda k: by_start[k][1])    # chỉ số, sắp theo end
    f = [0] * n
    best = None          # max(f[j] - end_j) trên các j đã kết thúc; None = chưa có ai
    ptr = 0
    ans = 0
    for i, (s, e, r) in enumerate(by_start):
        # gom mọi cuộc họp kết thúc <= s (chúng đều có start < s nên f đã tính xong)
        while ptr < n and by_start[order_end[ptr]][1] <= s:
            j = order_end[ptr]
            cand = f[j] - by_start[j][1]
            if best is None or cand > best:
                best = cand
            ptr += 1
        f[i] = r if best is None else r + s + best
        ans = max(ans, f[i])
    return ans

for meetings, expected in examples:
    result = max_earnings_sweep(meetings)
    print(meetings, "->", result, "| expected =", expected)
    assert result == expected

t0 = time.perf_counter()
assert max_earnings_sweep(big) == ans_big
print(f"n = 100000: khớp Cách 3, thời gian ~ {time.perf_counter() - t0:.2f} giây")

[[2, 5, 4], [6, 8, 3]] -> 8 | expected = 8
[[3, 5, 4], [4, 7, 8], [8, 10, 3]] -> 12 | expected = 12
[[1, 2, 2], [4, 5, 2], [7, 9, 3]] -> 11 | expected = 11


n = 100000: khớp Cách 3, thời gian ~ 0.09 giây


## Độ phức tạp của Cách 4

- Hai lần sắp xếp: `O(n log n)`.
- Vòng `for` chạy `n` lần; con trỏ `ptr` chỉ tiến, tổng cộng tối đa `n` bước cho **cả** quá trình → phần quét `O(n)`.
- Bộ nhớ: `O(n)` (`by_start`, `order_end`, `f`).

**Tổng: Time `O(n log n)` (do sắp xếp), Space `O(n)`.** So với Cách 3: không cần bisect, không cần mảng `pre_max`,
chỉ một biến `best`. Đổi lại phải giữ **hai** thứ tự sắp xếp cùng lúc.

> Ghi chú: một biến thể khác thay `order_end` bằng **min-heap** (`heapq`) theo `end`: đẩy `(end_i, f[i] − end_i)` vào heap
> sau khi tính `f[i]`, và trước mỗi `i` thì `heappop` mọi phần tử có `end <= start_i`. Cùng ý tưởng sweep, cùng `O(n log n)`.

# 11. Mô phỏng từng bước (Cách 3)

Dùng `TRACE_MEETINGS = [[1,7,6],[6,9,2],[0,2,3],[9,10,1],[2,5,1],[3,5,4]]`.
Ví dụ này được chọn vì có đủ các nhánh đáng chú ý: cuộc họp **không có ai đứng trước** (`p = 0`), **chạm mút**
(`end == start`), **cùng end**, và một cuộc họp có `revenue < độ dài` nên lịch tối ưu **bỏ qua** nó.

Sau khi sắp theo `end`: `ms = [[0,2,3],[2,5,1],[3,5,4],[1,7,6],[6,9,2],[9,10,1]]`, `ends = [2,5,5,7,9,10]`.

Ý nghĩa các cột: `p` = số cuộc họp kết thúc `<= s`; `pre_max[p]` = `max(f[j] − end_j)` trên `p` cuộc họp đó;
`f` = thu nhập tốt nhất của lịch kết thúc tại cuộc họp này; cột cuối là `pre_max` sau khi cập nhật.

| i | (s, e, r) | p | pre_max[p] | f | f − e | pre_max[i+1] | ans |
|---:|---|---:|---:|---|---:|---:|---:|
| 0 | (0, 2, 3) | 0 | -inf | 3 (một mình) | 1 | 1 | 3 |
| 1 | (2, 5, 1) | 1 | 1 | 1 + 2 + 1 = 4 | -1 | 1 | 4 |
| 2 | (3, 5, 4) | 1 | 1 | 4 + 3 + 1 = 8 | 3 | 3 | 8 |
| 3 | (1, 7, 6) | 0 | -inf | 6 (một mình) | -1 | 3 | 8 |
| 4 | (6, 9, 2) | 3 | 3 | 2 + 6 + 3 = 11 | 2 | 3 | 11 |
| 5 | (9, 10, 1) | 5 | 3 | 1 + 9 + 3 = 13 | 3 | 3 | 13 |

Đáp án `13` ứng với lịch `[0,2,3] → [3,5,4] → [9,10,1]`: doanh thu `3 + 4 + 1 = 8`, tiền rảnh `(3 − 2) + (9 − 5) = 5`.
Lịch này **bỏ** `[6,9,2]` (doanh thu 2 nhưng dài 3): nếu chèn vào, thu nhập chỉ còn `12`.

In [25]:
def trace_max_earnings(meetings):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [e for _, e, _ in ms]
    print("ms (theo end) =", ms)
    print("ends          =", ends)
    NEG = float("-inf")
    pre_max = [NEG] * (len(ms) + 1)
    ans = 0
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)
        print(f"--- i={i}: cuộc họp (s={s}, e={e}, r={r})")
        if p == 0:
            f = r
            print(f"    p = bisect_right(ends, {s}) = 0 -> không ai kết thúc <= {s} -> một mình: f = {f}")
        else:
            f = r + s + pre_max[p]
            shown = f"({pre_max[p]})" if pre_max[p] < 0 else f"{pre_max[p]}"
            print(f"    p = bisect_right(ends, {s}) = {p} -> pre_max[{p}] = {pre_max[p]} -> f = {r} + {s} + {shown} = {f}")
        pre_max[i + 1] = max(pre_max[i], f - e)
        ans = max(ans, f)
        print(f"    f - e = {f - e}; pre_max[{i + 1}] = max({pre_max[i]}, {f - e}) = {pre_max[i + 1]}; ans = {ans}")
    return ans

result = trace_max_earnings(TRACE_MEETINGS)
assert result == 13 == max_earnings_bruteforce(TRACE_MEETINGS)
print("Kết quả:", result, "(khớp brute force)")

ms (theo end) = [[0, 2, 3], [2, 5, 1], [3, 5, 4], [1, 7, 6], [6, 9, 2], [9, 10, 1]]
ends          = [2, 5, 5, 7, 9, 10]
--- i=0: cuộc họp (s=0, e=2, r=3)
    p = bisect_right(ends, 0) = 0 -> không ai kết thúc <= 0 -> một mình: f = 3
    f - e = 1; pre_max[1] = max(-inf, 1) = 1; ans = 3
--- i=1: cuộc họp (s=2, e=5, r=1)
    p = bisect_right(ends, 2) = 1 -> pre_max[1] = 1 -> f = 1 + 2 + 1 = 4
    f - e = -1; pre_max[2] = max(1, -1) = 1; ans = 4
--- i=2: cuộc họp (s=3, e=5, r=4)
    p = bisect_right(ends, 3) = 1 -> pre_max[1] = 1 -> f = 4 + 3 + 1 = 8
    f - e = 3; pre_max[3] = max(1, 3) = 3; ans = 8
--- i=3: cuộc họp (s=1, e=7, r=6)
    p = bisect_right(ends, 1) = 0 -> không ai kết thúc <= 1 -> một mình: f = 6
    f - e = -1; pre_max[4] = max(3, -1) = 3; ans = 8
--- i=4: cuộc họp (s=6, e=9, r=2)
    p = bisect_right(ends, 6) = 3 -> pre_max[3] = 3 -> f = 2 + 6 + 3 = 11
    f - e = 2; pre_max[5] = max(3, 2) = 3; ans = 11
--- i=5: cuộc họp (s=9, e=10, r=1)
    p = bisect_right(ends, 9) = 5 

## Mô phỏng Cách 4 (sweep) trên cùng ví dụ

Thứ tự theo `start`: `(0,2,3), (1,7,6), (2,5,1), (3,5,4), (6,9,2), (9,10,1)`.

| bước | cuộc họp (s, e, r) | gom thêm (end <= s) | best sau khi gom | f |
|---:|---|---|---:|---|
| 1 | (0, 2, 3) | – | None | 3 (một mình) |
| 2 | (1, 7, 6) | – | None | 6 (một mình) |
| 3 | (2, 5, 1) | (0,2,3): 3 − 2 = 1 | 1 | 1 + 2 + 1 = 4 |
| 4 | (3, 5, 4) | – | 1 | 4 + 3 + 1 = 8 |
| 5 | (6, 9, 2) | (2,5,1): −1 ; (3,5,4): 3 | 3 | 2 + 6 + 3 = 11 |
| 6 | (9, 10, 1) | (1,7,6): −1 ; (6,9,2): 2 | 3 | 1 + 9 + 3 = 13 |

Để ý bước 2: cuộc họp `(1,7,6)` được xử lý **trước** `(2,5,1)` dù kết thúc muộn hơn — Cách 4 đi theo `start`, không theo `end`.

In [26]:
def trace_sweep(meetings):
    by_start = sorted(meetings, key=lambda m: m[0])
    n = len(by_start)
    order_end = sorted(range(n), key=lambda k: by_start[k][1])
    f = [0] * n
    best, ptr = None, 0
    for i, (s, e, r) in enumerate(by_start):
        gathered = []
        while ptr < n and by_start[order_end[ptr]][1] <= s:
            j = order_end[ptr]
            cand = f[j] - by_start[j][1]
            gathered.append(f"{tuple(by_start[j])}:{cand}")
            best = cand if best is None else max(best, cand)
            ptr += 1
        f[i] = r if best is None else r + s + best
        print(f"bước {i + 1}: {tuple(by_start[i])} | gom: {gathered if gathered else '-'} | best = {best} | f = {f[i]}")
    return max(f)

print("Kết quả:", trace_sweep(TRACE_MEETINGS))

bước 1: (0, 2, 3) | gom: - | best = None | f = 3
bước 2: (1, 7, 6) | gom: - | best = None | f = 6
bước 3: (2, 5, 1) | gom: ['(0, 2, 3):1'] | best = 1 | f = 4
bước 4: (3, 5, 4) | gom: - | best = 1 | f = 8
bước 5: (6, 9, 2) | gom: ['(2, 5, 1):-1', '(3, 5, 4):3'] | best = 3 | f = 11
bước 6: (9, 10, 1) | gom: ['(1, 7, 6):-1', '(6, 9, 2):2'] | best = 3 | f = 13
Kết quả: 13


# 12. Phiên bản nộp LeetCode

Ta đưa Cách 3 vào đúng class và chữ ký của đề. Code nộp bài nên **gọn**, không `print`.

- `from typing import List` để chạy được ngoài LeetCode (trên LeetCode đã có sẵn).
- `from bisect import bisect_right`.
- Dùng `sorted(...)` thay vì `meetings.sort()` để không sửa dữ liệu đầu vào (trên LeetCode thì cả hai đều được chấp nhận).

In [27]:
from typing import List
from bisect import bisect_right

class Solution:
    def maxEarnings(self, meetings: List[List[int]]) -> int:
        ms = sorted(meetings, key=lambda m: m[1])
        ends = [m[1] for m in ms]
        pre_max = [float("-inf")] * (len(ms) + 1)
        ans = 0
        for i, (s, e, r) in enumerate(ms):
            p = bisect_right(ends, s)
            f = r if p == 0 else r + s + pre_max[p]
            ans = max(ans, f)
            pre_max[i + 1] = max(pre_max[i], f - e)
        return ans

In [28]:
sol = Solution()

# Ví dụ trong đề
assert sol.maxEarnings([[2, 5, 4], [6, 8, 3]]) == 8
assert sol.maxEarnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
assert sol.maxEarnings([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 11

# Edge cases
edge_cases = [
    ("một cuộc họp", [[2, 5, 4]], 4),
    ("chạm mút nối đuôi", [[0, 1, 1], [1, 2, 1], [2, 3, 1]], 3),
    ("bỏ cuộc họp giữa", [[0, 1, 1], [1, 10, 1], [10, 11, 1]], 11),
    ("tất cả chồng nhau", [[0, 10, 5], [1, 9, 7], [2, 8, 6]], 7),
    ("trùng lặp hoàn toàn", [[1, 4, 5], [1, 4, 5], [1, 4, 5]], 5),
    ("lồng nhau", [[0, 10, 5], [2, 3, 4], [5, 6, 4]], 10),
    ("cùng end", [[2, 6, 3], [4, 6, 3], [0, 1, 1]], 7),
    ("giá trị lớn", [[0, 1, 10 ** 9], [10 ** 9 - 1, 10 ** 9, 10 ** 9]], 2_999_999_998),
    ("start = 0, một cuộc họp dài", [[0, 10 ** 9, 10 ** 9]], 10 ** 9),
]
for name, meetings, expected in edge_cases:
    result = sol.maxEarnings(meetings)
    status = "PASS" if result == expected else "FAIL"
    print(f"{name:<28} -> {result:>13} | expected = {expected:>13} | {status}")
    assert result == expected
    if len(meetings) <= 8:
        assert result == max_earnings_bruteforce(meetings)

# Không làm thay đổi input
data = [[6, 8, 3], [2, 5, 4]]
sol.maxEarnings(data)
assert data == [[6, 8, 3], [2, 5, 4]]
print("Tất cả test của class Solution đều PASS!")

một cuộc họp                 ->             4 | expected =             4 | PASS
chạm mút nối đuôi            ->             3 | expected =             3 | PASS
bỏ cuộc họp giữa             ->            11 | expected =            11 | PASS
tất cả chồng nhau            ->             7 | expected =             7 | PASS
trùng lặp hoàn toàn          ->             5 | expected =             5 | PASS
lồng nhau                    ->            10 | expected =            10 | PASS
cùng end                     ->             7 | expected =             7 | PASS
giá trị lớn                  ->    2999999998 | expected =    2999999998 | PASS
start = 0, một cuộc họp dài  ->    1000000000 | expected =    1000000000 | PASS
Tất cả test của class Solution đều PASS!


# 13. Kiểm tra chéo ngẫu nhiên (stress test)

Ví dụ trong đề chỉ có 3 ca — quá ít để tin một bài Hard. Ta sinh **nhiều input nhỏ ngẫu nhiên** (seed cố định để chạy lại
ra cùng kết quả) và so **mọi cách** với brute force. Input nhỏ (`n <= 7`, thời gian `<= 12`) để brute force chạy nhanh, nhưng
vẫn tạo ra đủ kiểu: chạm mút, cùng end, lồng nhau, trùng lặp.

(Trước khi viết notebook, các lời giải đã được kiểm tra trên 3000 ca ngẫu nhiên và 1 ca `n = 10^5` — cell dưới là bản rút gọn.)

In [29]:
import random

def random_small_case(rng):
    n = rng.randint(1, 7)
    T = rng.choice([5, 8, 12])
    meetings = []
    for _ in range(n):
        s = rng.randint(0, T - 1)
        e = rng.randint(s + 1, T)
        meetings.append([s, e, rng.randint(1, rng.choice([3, 10]))])
    return meetings

approaches = {
    "dp_quadratic": max_earnings_dp_quadratic,
    "bisect": max_earnings_bisect,
    "sweep": max_earnings_sweep,
    "Solution": lambda m: Solution().maxEarnings(m),
}

rng = random.Random(4068)
num_cases = 400
for _ in range(num_cases):
    meetings = random_small_case(rng)
    expected = max_earnings_bruteforce(meetings)
    for name, fn in approaches.items():
        got = fn(meetings)
        assert got == expected, (name, meetings, got, expected)
print(f"{num_cases} ca ngẫu nhiên: cả {len(approaches)} cách đều khớp brute force.")

400 ca ngẫu nhiên: cả 4 cách đều khớp brute force.


# 14. Vì sao thuật toán đúng?

## (1) Đáp án = max của `f[i]`

Mọi lịch hợp lệ khác rỗng có đúng một cuộc họp kết thúc muộn nhất (cũng là cuộc họp cuối khi xếp theo `start`,
vì trong lịch hợp lệ thứ tự theo `start` trùng thứ tự theo `end`). Chia mọi lịch theo "cuộc họp cuối" → mỗi lịch rơi vào
đúng một nhóm `i`, và `f[i]` là lịch tốt nhất của nhóm đó. Vậy lịch tốt nhất toàn cục = `max(f[i])`.

## (2) Công thức của `f[i]` đúng (cấu trúc con tối ưu)

- **Không bỏ sót:** xét lịch tốt nhất kết thúc tại `i`. Nếu chỉ có `i` → giá trị `revenue_i`. Nếu không, gọi `j` là cuộc họp
  ngay trước `i`: `end_j <= start_i`. Bỏ `i`, phần còn lại là lịch hợp lệ kết thúc tại `j`, có giá trị `<= f[j]`.
  Vậy lịch tốt nhất `<= f[j] + revenue_i + (start_i − end_j)` — một trong các phương án công thức xét tới.
- **Không trả nhầm:** ngược lại, lấy lịch đạt `f[j]` (mọi cuộc họp trong đó kết thúc `<= end_j <= start_i`) rồi nối thêm `i`
  → vẫn hợp lệ, và giá trị đúng bằng `f[j] + revenue_i + (start_i − end_j)`. Nên mọi phương án công thức xét đều **đạt được thật**.

## (3) Tách biến và tiền tố (Cách 3)

`f[j] + revenue_i + (start_i − end_j) = (revenue_i + start_i) + (f[j] − end_j)` → chỉ cần `max(f[j] − end_j)`.
Trong `ms` sắp theo `end`, tập `{j : end_j <= start_i}` là tiền tố độ dài `p = bisect_right(ends, start_i)`, và `p <= i`
(vì `end_j <= start_i < end_i`).

**Invariant:** ngay trước khi xử lý `i`, `pre_max[k] = max(f[j] − end_j : j < k)` với mọi `k <= i` (`pre_max[0] = -inf`).
Cập nhật `pre_max[i+1] = max(pre_max[i], f[i] − end_i)` giữ invariant cho bước sau. Vì `p <= i`, `pre_max[p]` đã đúng khi cần.

## (4) Sweep (Cách 4)

`j` hợp lệ cho `i` ⇒ `start_j < end_j <= start_i` ⇒ `j` đứng trước `i` theo `start`, nên `f[j]` đã tính. Con trỏ `ptr`
gom đúng các `j` có `end_j <= start_i`; vì `start_i` không giảm, không `j` nào cần "bỏ ra" sau khi đã gom.

**Invariant:** trước khi tính `f[i]`, `best = max(f[j] − end_j : end_j <= start_i)` (hoặc `None` nếu tập rỗng).

## (5) Vì sao không cần `max(một mình, nối)` khi đã có `j`?

`f[j] − end_j + start_i >= f[j] >= 1` (vì `start_i >= end_j` và mọi `revenue >= 1`), nên `revenue_i + start_i + best > revenue_i`.
Viết thêm `max(r, ...)` cũng không sai, chỉ là thừa.

# 15. Độ phức tạp và so sánh các cách

Gọi `n` = số cuộc họp (`n <= 10^5`).

| Cách | Ý tưởng chính | Thời gian | Bộ nhớ phụ | Dùng được với n = 10^5? |
|---|---|---|---|---|
| 1. Brute force | thử mọi tập con, kiểm mọi cặp | `O(2^n · n²)` | `O(n)` | Không (chỉ n ≲ 15) |
| 2. DP `O(n²)` | `f[i]` = max qua mọi `j` trước | `O(n²)` | `O(n)` | Không (~5·10^9 bước) |
| 3. DP + bisect + prefix max | tách biến, `j` hợp lệ là tiền tố | `O(n log n)` | `O(n)` | **Có** |
| 4. DP + sweep hai con trỏ | duyệt theo start, một biến `best` | `O(n log n)` (sắp xếp) | `O(n)` | **Có** |

Nhận xét:

- Từ Cách 1 → 2: thay "thử mọi tập con" bằng "nhớ lịch tốt nhất kết thúc tại mỗi cuộc họp" (DP).
- Từ Cách 2 → 3/4: thay "tính lại max cho từng `i`" bằng **tách biến** + "max của tiền tố" tính dồn.
- Về Python trên LeetCode: Cách 3 với `n = 10^5` chỉ tốn cỡ vài phần mười giây, rất an toàn. Nếu muốn nhanh hơn nữa, có thể bỏ
  `enumerate`/unpacking hoặc dùng Cách 4, nhưng không cần thiết.

# 16. Những lỗi thường gặp

## Lỗi 1 — Dùng `bisect_left` thay vì `bisect_right`

`bisect_left(ends, s)` đếm số `end < s` → **bỏ sót** cuộc họp kết thúc đúng lúc `s` (chạm mút, vẫn hợp lệ với khoảng nửa mở).
Sửa: dùng `bisect_right(ends, s)` (đếm `end <= s`).

## Lỗi 2 — Khởi tạo `pre_max` bằng `0` (cuộc họp "ảo" ở thời điểm 0)

Nếu `pre_max = [0] * (n + 1)` thì cuộc họp nào cũng được "nối" sau một cuộc họp ảo thu nhập 0 kết thúc lúc 0 →
cộng nhầm `start_i` đồng tiền rảnh **trước cuộc họp đầu tiên**. Đề nói rõ: không có tiền rảnh trước cuộc họp sớm nhất.
Sửa: `pre_max[0] = -inf` và xử lý riêng `p == 0`.

## Lỗi 3 — Tham lam "chọn càng nhiều cuộc họp càng tốt"

Thuật toán chọn hoạt động kinh điển (activity selection: sắp theo end, gặp cuộc họp không chồng thì lấy) tối đa **số lượng**,
không tối đa **thu nhập**. Chèn một cuộc họp vào giữa làm thay đổi `revenue − độ dài`, có thể âm (Bước đột phá 2).

In [30]:
from bisect import bisect_left, bisect_right

def wrong_bisect_left(meetings):
    ms = sorted(meetings, key=lambda m: m[1]); ends = [m[1] for m in ms]
    pre_max = [float("-inf")] * (len(ms) + 1); ans = 0
    for i, (s, e, r) in enumerate(ms):
        p = bisect_left(ends, s)                       # SAI: bỏ sót end == s
        f = r if p == 0 else r + s + pre_max[p]
        ans = max(ans, f); pre_max[i + 1] = max(pre_max[i], f - e)
    return ans

def wrong_zero_sentinel(meetings):
    ms = sorted(meetings, key=lambda m: m[1]); ends = [m[1] for m in ms]
    pre_max = [0] * (len(ms) + 1); ans = 0               # SAI: 0 thay vì -inf
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)
        f = r + s + pre_max[p]                         # luôn "nối", kể cả khi p == 0
        ans = max(ans, f); pre_max[i + 1] = max(pre_max[i], f - e)
    return ans

def wrong_greedy_take_all(meetings):
    ms = sorted(meetings, key=lambda m: m[1]); chosen = []; last_end = -1
    for s, e, r in ms:
        if s >= last_end:                              # SAI: cứ không chồng là lấy
            chosen.append([s, e, r]); last_end = e
    return earnings_of(chosen)

tests = [
    ("Lỗi 1", wrong_bisect_left, [[1, 3, 2], [3, 5, 2]]),
    ("Lỗi 2", wrong_zero_sentinel, [[5, 6, 1]]),
    ("Lỗi 2", wrong_zero_sentinel, [[2, 5, 4], [6, 8, 3]]),
    ("Lỗi 3", wrong_greedy_take_all, [[0, 1, 1], [1, 10, 1], [10, 11, 1]]),
]
for label, fn, m in tests:
    print(f"{label}: {m} -> code sai = {fn(m)}, đúng = {max_earnings_bisect(m)}")

Lỗi 1: [[1, 3, 2], [3, 5, 2]] -> code sai = 2, đúng = 4
Lỗi 2: [[5, 6, 1]] -> code sai = 6, đúng = 1
Lỗi 2: [[2, 5, 4], [6, 8, 3]] -> code sai = 10, đúng = 8
Lỗi 3: [[0, 1, 1], [1, 10, 1], [10, 11, 1]] -> code sai = 3, đúng = 11


Giải thích kết quả:

- **Lỗi 1:** `[[1,3,2],[3,5,2]]` — đúng phải nối hai cuộc họp (`2 + 2 + 0 = 4`), `bisect_left` không thấy `[1,3)` nên chỉ được `2`.
- **Lỗi 2:** `[[5,6,1]]` — code sai cộng thêm `5` đơn vị "rảnh" từ thời điểm 0 tới 5 → `6` thay vì `1`. Ví dụ 1 của đề cũng sai (`10` thay vì `8`).
- **Lỗi 3:** tham lam lấy cả ba → `3`, trong khi bỏ cuộc họp giữa được `11`.

## Lỗi 4 — Sắp theo `start` nhưng vẫn bisect trên `ends`

Nếu sắp theo `start`, danh sách `ends` **không** tăng dần nữa → `bisect` trả kết quả vô nghĩa, và tệ hơn, các `j` có
`end_j <= start_i` không còn là một tiền tố. Code có thể "nối" hai cuộc họp **chồng nhau** và trả về số **lớn hơn đáp án thật**.

## Lỗi 5 — Quên trường hợp `p == 0` / quên cập nhật `ans` cho cuộc họp đơn lẻ

Nếu chỉ cập nhật `ans` bên trong nhánh "nối được", thì một input chỉ có cuộc họp đơn lẻ (ví dụ `n = 1`) sẽ trả về `0`.
Đề yêu cầu tập con **khác rỗng**, nên đáp án luôn `>= max(revenue) >= 1`.

In [31]:
def wrong_sort_by_start(meetings):
    ms = sorted(meetings, key=lambda m: m[0])          # SAI: sắp theo start
    ends = [m[1] for m in ms]                          # ends không còn tăng dần!
    pre_max = [float("-inf")] * (len(ms) + 1); ans = 0
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)
        f = r if p == 0 else r + s + pre_max[p]
        ans = max(ans, f); pre_max[i + 1] = max(pre_max[i], f - e)
    return ans

def wrong_forget_alone(meetings):
    ms = sorted(meetings, key=lambda m: m[1]); ends = [m[1] for m in ms]
    pre_max = [float("-inf")] * (len(ms) + 1); ans = 0
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)
        f = r
        if p > 0:
            f = r + s + pre_max[p]
            ans = max(ans, f)                          # SAI: chỉ cập nhật ans khi nối được
        pre_max[i + 1] = max(pre_max[i], f - e)
    return ans

m4 = [[0, 3, 5], [1, 2, 1], [2, 3, 3]]
print("Lỗi 4:", m4, "-> code sai =", wrong_sort_by_start(m4), ", đúng =", max_earnings_bisect(m4))
print("       ends sau khi sắp theo start:", [m[1] for m in sorted(m4, key=lambda m: m[0])], "(không tăng dần)")
m5 = [[2, 5, 4]]
print("Lỗi 5:", m5, "-> code sai =", wrong_forget_alone(m5), ", đúng =", max_earnings_bisect(m5))

Lỗi 4: [[0, 3, 5], [1, 2, 1], [2, 3, 3]] -> code sai = 7 , đúng = 5
       ends sau khi sắp theo start: [3, 2, 3] (không tăng dần)
Lỗi 5: [[2, 5, 4]] -> code sai = 0 , đúng = 4


Ở Lỗi 4, code sai trả `7`: nó "nối" `[2,3,3]` sau `[0,3,5]` (`5 + 3 + (2 − 3) = 7`) — hai cuộc họp này **chồng nhau** trên `[2,3)`!
Đáp án đúng là `5` (chọn `[0,3,5]` một mình, hoặc `[1,2,1] → [2,3,3]` cũng chỉ được `4`).

## Lỗi 6 — Những lỗi "ngoài thuật toán"

- **Tràn số** khi dịch sang C++/Java: dùng `long long` / `long` cho `f`, `pre_max`, `ans` (ca "giá trị lớn" ở mục 12 cho `2 999 999 998 > 2^31 − 1`).
- **Sửa input** bằng `meetings.sort()` rồi dùng lại `meetings` ở chỗ khác với giả định thứ tự cũ.
- **Nhầm khoảng đóng/mở**: nếu một bài khác dùng khoảng **đóng** `[start, end]` (ví dụ LeetCode 2054), điều kiện đứng trước là
  `end_j < start_i` → phải dùng `bisect_left`. Luôn đọc kỹ đề!

## Edge cases cần nhớ

| Trường hợp | Ví dụ | Đáp án |
|---|---|---:|
| Một cuộc họp | `[[2,5,4]]` | 4 |
| Nối đuôi chạm mút | `[[0,1,1],[1,2,1],[2,3,1]]` | 3 |
| Tất cả chồng nhau | `[[0,10,5],[1,9,7],[2,8,6]]` | 7 |
| Cuộc họp trùng lặp | `[[1,4,5]] × 3` | 5 |
| Cùng end | `[[2,6,3],[4,6,3],[0,1,1]]` | 7 |
| Giá trị lớn | `[[0,1,10^9],[10^9−1,10^9,10^9]]` | 2 999 999 998 |

(Tất cả đã được assert ở cell kiểm tra `class Solution`.)

# 17. Dấu hiệu nhận dạng và các bài liên quan

## Dấu hiệu nhận dạng

1. **"Chọn một tập con các khoảng thời gian không chồng nhau, tối đa hoá tổng giá trị"** → họ bài *weighted interval scheduling*:
   sắp theo `end` + DP "kết thúc tại `i`" + tìm `j` bằng binary search.
2. **Giá trị của một bước phụ thuộc vào cặp kề nhau `(j, i)` dưới dạng `A(i) + B(j)`** → tách biến, giữ sẵn `max B(j)`
   (prefix max / một biến `best`). Ở bài này `A(i) = revenue_i + start_i`, `B(j) = f[j] − end_j`.
3. **Toạ độ rất lớn (`10^9`) nhưng số phần tử vừa phải (`10^5`)** → đừng lập mảng theo thời gian; làm trên danh sách đã sắp xếp + `bisect`
   (hoặc nén toạ độ).
4. **Điều kiện "`end_j <= start_i`"** → tập `j` hợp lệ là **tiền tố** khi sắp theo `end`. Khoảng nửa mở → `bisect_right`; khoảng đóng → `bisect_left`.
5. **Duyệt theo một thứ tự, "gom" theo thứ tự khác** (start vs end) → nghĩ tới sweep hai con trỏ hoặc heap.

## Bài liên quan

| Bài | Độ khó | Liên hệ |
|---|---|---|
| 1235. Maximum Profit in Job Scheduling | Hard | Đúng khuôn này nhưng **không** có tiền rảnh: `f[i] = profit_i + max f[j]` |
| 2008. Maximum Earnings From Taxi | Medium | Thu nhập mỗi chuyến `end − start + tip`, cùng khuôn DP + bisect |
| 2054. Two Best Non-Overlapping Events | Medium | Chọn tối đa 2 sự kiện, khoảng **đóng**; prefix max + bisect |
| 1751. Maximum Number of Events That Can Be Attended II | Hard | Thêm chiều "tối đa `k` sự kiện": `f[i][c]` |
| 2830. Maximize the Profit as the Salesman | Medium | Chọn đoạn không chồng nhau trên trục số |
| 646. Maximum Length of Pair Chain / 435. Non-overlapping Intervals | Medium | Chỉ đếm số lượng → tham lam theo `end` là đủ |

Cell dưới áp dụng **đúng khuôn của Cách 3** cho bài 1235 (bỏ phần tiền rảnh: không còn `+ s` và `− e`).

In [32]:
from bisect import bisect_right

def job_scheduling(startTime, endTime, profit):
    jobs = sorted(zip(startTime, endTime, profit), key=lambda t: t[1])
    ends = [e for _, e, _ in jobs]
    pre_max = [float("-inf")] * (len(jobs) + 1)      # pre_max[k] = max f[j], j < k
    ans = 0
    for i, (s, e, p_) in enumerate(jobs):
        p = bisect_right(ends, s)
        f = p_ if p == 0 else p_ + pre_max[p]
        ans = max(ans, f)
        pre_max[i + 1] = max(pre_max[i], f)
    return ans

print(job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]))   # 120 (ví dụ 1 của bài 1235)

120


# 18. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. **Đừng** kéo xuống phần đáp án trước khi tự thử!
Các hàm TODO dưới đây chưa được gọi; tự làm xong rồi mới bỏ comment các dòng test.

## Bài 1 — Đoán output (không chạy code)

Hãy đoán kết quả của từng dòng:

```python
bisect_left([2, 5, 5, 7], 5)      # (a)
bisect_right([2, 5, 5, 7], 5)     # (b)
bisect_right([2, 5, 5, 7], 1)     # (c)
max_earnings_bisect([[1, 3, 2], [3, 5, 2]])                 # (d)
max_earnings_bisect([[0, 5, 1], [2, 3, 9]])                 # (e)
max_earnings_bisect([[0, 1, 1], [1, 10, 1], [10, 11, 1]])   # (f)
```

## Bài 2 — Tính tay bảng DP

Với `meetings = [[0,2,1],[3,4,2],[1,6,5],[5,8,3]]`, hãy tự lập bảng (sau khi sắp theo `end`):

| i | (s, e, r) | p | pre_max[p] | f | f − e | pre_max[i+1] | ans |
|---:|---|---:|---:|---:|---:|---:|---:|

rồi cho biết đáp án và lịch họp đạt đáp án đó. Kiểm tra lại bằng `trace_max_earnings` ở phần đáp án.

## Bài 3 — Điền TODO cho Cách 3

Hoàn thiện hàm dưới đây (sắp theo end → bisect_right → prefix max).

In [33]:
def exercise_3_max_earnings(meetings):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    pre_max = [float("-inf")] * (len(ms) + 1)
    ans = 0
    for i, (s, e, r) in enumerate(ms):
        # TODO 1: p = số cuộc họp có end <= s
        # TODO 2: nếu p == 0 thì f = r, ngược lại f = r + s + pre_max[p]
        # TODO 3: cập nhật ans
        # TODO 4: cập nhật pre_max[i + 1]
        pass
    return ans

# Tự làm xong rồi mới mở test:
# assert exercise_3_max_earnings([[2, 5, 4], [6, 8, 3]]) == 8
# assert exercise_3_max_earnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
# assert exercise_3_max_earnings([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 11

## Bài 4 — Viết lại theo chiều ngược: "lịch tốt nhất **bắt đầu** tại `i`"

Đặt `g[i]` = thu nhập lớn nhất của lịch hợp lệ mà **cuộc họp đầu tiên** là `i`. Hãy:

1. Viết công thức của `g[i]` theo các `g[j]` của những cuộc họp `j` có `start_j >= end_i` (đứng **sau** `i`).
2. Tách biến để thấy cần giữ `max(g[j] + start_j)` trên một **hậu tố** (suffix) khi sắp theo `start`.
3. Cài đặt `O(n log n)` bằng `bisect_left` trên danh sách `starts`.

Đáp án vẫn phải khớp `max_earnings_bisect` trên mọi input.

In [34]:
def exercise_4_max_earnings_suffix(meetings):
    # TODO: sắp theo start, duyệt i từ phải sang trái,
    #       q = vị trí đầu tiên có start >= end_i,
    #       g = r nếu không có ai phía sau, ngược lại g = r - e + suf_max[q],
    #       suf_max[i] = max(suf_max[i + 1], g + s)
    raise NotImplementedError("Hãy tự viết lại theo chiều ngược")

# assert exercise_4_max_earnings_suffix([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 11
# assert exercise_4_max_earnings_suffix([[1, 7, 6], [6, 9, 2], [0, 2, 3], [9, 10, 1], [2, 5, 1], [3, 5, 4]]) == 13

## Bài 5 — Biến thể: thời gian rảnh bị **phạt** `c` đồng mỗi đơn vị

Giữ nguyên đề, nhưng mỗi đơn vị thời gian rảnh giữa hai cuộc họp kề nhau làm bạn **mất** `c` đồng (`c >= 0` là số nguyên).
Thu nhập = tổng revenue − `c` × tổng thời gian rảnh. Viết hàm `O(n log n)`.

Câu hỏi suy nghĩ trước khi code:

- Tách biến thế nào? (gợi ý: `− c·(start_i − end_j) = − c·start_i + c·end_j`)
- Bước đột phá 2 ("nối luôn có lời") **còn đúng không**? Nếu không, công thức `f[i]` phải sửa ở đâu?

Ví dụ: `meetings = [[0,1,5],[10,11,5]]`, `c = 1` → nối hai cuộc họp mất `9` → chỉ được `1`, nên chọn một cuộc họp: đáp án `5`.

In [35]:
def exercise_5_max_earnings_penalty(meetings, c):
    # TODO: giống Cách 3 nhưng:
    #   - key của j là f[j] + c * end_j
    #   - phương án nối: r - c * s + pre_max[p]
    #   - nhớ so sánh với phương án "một mình"!
    raise NotImplementedError

# assert exercise_5_max_earnings_penalty([[0, 1, 5], [10, 11, 5]], 1) == 5
# assert exercise_5_max_earnings_penalty([[0, 1, 5], [2, 3, 5]], 1) == 9
# assert exercise_5_max_earnings_penalty([[1, 2, 2], [4, 5, 2], [7, 9, 3]], 0) == 7

## Bài 6 — Mini challenge: trả về **lịch họp** tối ưu

Viết hàm trả về `(thu_nhập_lớn_nhất, danh_sách_cuộc_họp_được_chọn)` theo thứ tự thời gian.

Yêu cầu: `O(n log n)`; danh sách trả về phải hợp lệ và `earnings_of(danh_sách) == thu_nhập_lớn_nhất`.

Ví dụ: với `TRACE_MEETINGS`, một đáp án là `(13, [[0, 2, 3], [3, 5, 4], [9, 10, 1]])`.

In [36]:
def exercise_6_best_schedule(meetings):
    # TODO:
    #   - ngoài pre_max, lưu thêm pre_arg[k] = chỉ số j đạt max trong k phần tử đầu
    #   - lưu parent[i] = j tốt nhất đứng trước i (hoặc -1 nếu i một mình)
    #   - lần ngược parent từ cuộc họp cuối tốt nhất để dựng lại lịch
    raise NotImplementedError

# best, schedule = exercise_6_best_schedule(TRACE_MEETINGS)
# assert best == 13 and earnings_of(schedule) == 13

# 19. Gợi ý (3 mức: hướng tư duy → cấu trúc / cú pháp → gần-code)

**Bài 1**
- Mức 1: `bisect_left` đếm phần tử `< x`, `bisect_right` đếm phần tử `<= x`.
- Mức 2: (d) chạm mút có được nối không? (e) hai cuộc họp có chồng nhau không? (f) xem lại mục 2.3.
- Mức 3: (d) `2 + 2 + 0`; (e) chỉ chọn được một; (f) bỏ cuộc họp giữa.

**Bài 2**
- Mức 1: sắp theo `end` trước: `ends = [2, 4, 6, 8]`.
- Mức 2: mỗi dòng: `p = bisect_right(ends, s)` → `f` → `f − e` → `pre_max[i+1]`.
- Mức 3: dòng `(1,6,5)` có `p = 0` (không ai kết thúc `<= 1`); dòng `(5,8,3)` có `p = 2`.

**Bài 3**
- Mức 1: chép lại 4 bước của Cách 3, chú ý thứ tự "tính `f` → cập nhật `ans` → cập nhật `pre_max`".
- Mức 2: `p = bisect_right(ends, s)`; `f = r if p == 0 else r + s + pre_max[p]`.
- Mức 3: `ans = max(ans, f)`; `pre_max[i + 1] = max(pre_max[i], f - e)`; nhớ `from bisect import bisect_right`.

**Bài 4**
- Mức 1: nối `j` **sau** `i`: `g[i] = r_i + (start_j − end_i) + g[j]` → phần của `j` là `g[j] + start_j`.
- Mức 2: sắp theo `start`; các `j` có `start_j >= end_i` là **hậu tố** bắt đầu tại `q = bisect_left(starts, end_i)`.
- Mức 3: duyệt `i` từ `n − 1` về `0`; `suf_max = [-inf] * (n + 1)`; `g = r if q == n else r - e + suf_max[q]`; `suf_max[i] = max(suf_max[i + 1], g + s)`.

**Bài 5**
- Mức 1: lần này nối có thể **lỗ**, nên phải `max` với phương án một mình.
- Mức 2: `f[i] = max(r, r − c·s + max(f[j] + c·end_j))`.
- Mức 3: `pre_max[i + 1] = max(pre_max[i], f + c * e)`; khi `p > 0`: `f = max(r, r - c * s + pre_max[p])`.

**Bài 6**
- Mức 1: muốn dựng lại lịch thì mỗi `f[i]` phải nhớ "nó được nối sau ai".
- Mức 2: song song với `pre_max` giữ `pre_arg` (chỉ số đạt max); `parent[i] = pre_arg[p]` khi `p > 0`.
- Mức 3: tìm `i*` có `f` lớn nhất; `while i* != -1: schedule.append(ms[i*]); i* = parent[i*]`; cuối cùng đảo ngược `schedule`.

# Đáp án tham khảo

Chỉ xem sau khi đã tự làm. Mỗi đáp án đều có `assert` để tự kiểm chứng.

## Đáp án Bài 1

(a) `1` (b) `3` (c) `0` (d) `4` (e) `9` (f) `11`

In [37]:
from bisect import bisect_left, bisect_right

print("(a)", bisect_left([2, 5, 5, 7], 5))
print("(b)", bisect_right([2, 5, 5, 7], 5))
print("(c)", bisect_right([2, 5, 5, 7], 1))
print("(d)", max_earnings_bisect([[1, 3, 2], [3, 5, 2]]))
print("(e)", max_earnings_bisect([[0, 5, 1], [2, 3, 9]]))
print("(f)", max_earnings_bisect([[0, 1, 1], [1, 10, 1], [10, 11, 1]]))
assert [bisect_left([2, 5, 5, 7], 5), bisect_right([2, 5, 5, 7], 5), bisect_right([2, 5, 5, 7], 1)] == [1, 3, 0]

(a) 1
(b) 3
(c) 0
(d) 4
(e) 9
(f) 11


## Đáp án Bài 2

| i | (s, e, r) | p | pre_max[p] | f | f − e | pre_max[i+1] | ans |
|---:|---|---:|---:|---|---:|---:|---:|
| 0 | (0, 2, 1) | 0 | -inf | 1 (một mình) | -1 | -1 | 1 |
| 1 | (3, 4, 2) | 1 | -1 | 2 + 3 − 1 = 4 | 0 | 0 | 4 |
| 2 | (1, 6, 5) | 0 | -inf | 5 (một mình) | -1 | 0 | 5 |
| 3 | (5, 8, 3) | 2 | 0 | 3 + 5 + 0 = 8 | 0 | 0 | 8 |

Đáp án `8`: lịch `[0,2,1] → [3,4,2] → [5,8,3]` (doanh thu `6`, rảnh `1 + 1 = 2`).

In [38]:
ex2 = [[0, 2, 1], [3, 4, 2], [1, 6, 5], [5, 8, 3]]
assert trace_max_earnings(ex2) == 8 == max_earnings_bruteforce(ex2)
assert earnings_of([[0, 2, 1], [3, 4, 2], [5, 8, 3]]) == 8

ms (theo end) = [[0, 2, 1], [3, 4, 2], [1, 6, 5], [5, 8, 3]]
ends          = [2, 4, 6, 8]
--- i=0: cuộc họp (s=0, e=2, r=1)
    p = bisect_right(ends, 0) = 0 -> không ai kết thúc <= 0 -> một mình: f = 1
    f - e = -1; pre_max[1] = max(-inf, -1) = -1; ans = 1
--- i=1: cuộc họp (s=3, e=4, r=2)
    p = bisect_right(ends, 3) = 1 -> pre_max[1] = -1 -> f = 2 + 3 + (-1) = 4
    f - e = 0; pre_max[2] = max(-1, 0) = 0; ans = 4
--- i=2: cuộc họp (s=1, e=6, r=5)
    p = bisect_right(ends, 1) = 0 -> không ai kết thúc <= 1 -> một mình: f = 5
    f - e = -1; pre_max[3] = max(0, -1) = 0; ans = 5
--- i=3: cuộc họp (s=5, e=8, r=3)
    p = bisect_right(ends, 5) = 2 -> pre_max[2] = 0 -> f = 3 + 5 + 0 = 8
    f - e = 0; pre_max[4] = max(0, 0) = 0; ans = 8


## Đáp án Bài 3

In [39]:
from bisect import bisect_right

def answer_3_max_earnings(meetings):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    pre_max = [float("-inf")] * (len(ms) + 1)
    ans = 0
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)                   # TODO 1
        f = r if p == 0 else r + s + pre_max[p]     # TODO 2
        ans = max(ans, f)                           # TODO 3
        pre_max[i + 1] = max(pre_max[i], f - e)     # TODO 4
    return ans

assert answer_3_max_earnings([[2, 5, 4], [6, 8, 3]]) == 8
assert answer_3_max_earnings([[3, 5, 4], [4, 7, 8], [8, 10, 3]]) == 12
assert answer_3_max_earnings([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 11
print("Bài 3: PASS")

Bài 3: PASS


## Đáp án Bài 4

Công thức: `g[i] = r_i` (một mình) hoặc `g[i] = r_i − end_i + max(g[j] + start_j : start_j >= end_i)`.
Sắp theo `start`, các `j` hợp lệ là hậu tố từ `q = bisect_left(starts, end_i)` (`start_j >= end_i` → `bisect_left`).

In [40]:
from bisect import bisect_left
import random

def answer_4_max_earnings_suffix(meetings):
    ms = sorted(meetings, key=lambda m: m[0])
    starts = [m[0] for m in ms]
    n = len(ms)
    suf_max = [float("-inf")] * (n + 1)     # suf_max[k] = max(g[j] + start_j), j >= k
    ans = 0
    for i in range(n - 1, -1, -1):
        s, e, r = ms[i]
        q = bisect_left(starts, e)           # cuộc họp đầu tiên có start >= e
        g = r if q == n else r - e + suf_max[q]
        ans = max(ans, g)
        suf_max[i] = max(suf_max[i + 1], g + s)
    return ans

assert answer_4_max_earnings_suffix([[1, 2, 2], [4, 5, 2], [7, 9, 3]]) == 11
assert answer_4_max_earnings_suffix(TRACE_MEETINGS) == 13
rng = random.Random(4)
for _ in range(200):
    m = random_small_case(rng)
    assert answer_4_max_earnings_suffix(m) == max_earnings_bruteforce(m)
print("Bài 4: PASS (cả 200 ca ngẫu nhiên)")

Bài 4: PASS (cả 200 ca ngẫu nhiên)


## Đáp án Bài 5

Tách biến: `f[j] + r_i − c·(start_i − end_j) = (r_i − c·start_i) + (f[j] + c·end_j)`.
Bước đột phá 2 **không còn đúng**: nối có thể lỗ, nên phải lấy `max` với phương án một mình.
Kiểm chứng bằng một brute force tổng quát với hệ số rảnh `rate` (`rate = −c`).

In [41]:
from bisect import bisect_right
from itertools import combinations
import random

def answer_5_max_earnings_penalty(meetings, c):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    pre_max = [float("-inf")] * (len(ms) + 1)    # max(f[j] + c * end_j)
    ans = None
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)
        f = r
        if p > 0:
            f = max(r, r - c * s + pre_max[p])   # nối có thể lỗ -> so với một mình
        ans = f if ans is None else max(ans, f)
        pre_max[i + 1] = max(pre_max[i], f + c * e)
    return ans

def bruteforce_rate(meetings, rate):
    best = None
    for k in range(1, len(meetings) + 1):
        for chosen in combinations(meetings, k):
            if any(overlaps(a, b) for a, b in combinations(chosen, 2)):
                continue
            order = sorted(chosen, key=lambda m: m[0])
            value = sum(m[2] for m in order) + rate * sum(order[t + 1][0] - order[t][1] for t in range(len(order) - 1))
            best = value if best is None else max(best, value)
    return best

assert answer_5_max_earnings_penalty([[0, 1, 5], [10, 11, 5]], 1) == 5
assert answer_5_max_earnings_penalty([[0, 1, 5], [2, 3, 5]], 1) == 9
assert answer_5_max_earnings_penalty([[1, 2, 2], [4, 5, 2], [7, 9, 3]], 0) == 7
rng = random.Random(5)
for _ in range(200):
    m = random_small_case(rng)
    c = rng.randint(0, 3)
    assert answer_5_max_earnings_penalty(m, c) == bruteforce_rate(m, -c)
print("Bài 5: PASS (cả 200 ca ngẫu nhiên với c = 0..3)")

Bài 5: PASS (cả 200 ca ngẫu nhiên với c = 0..3)


## Đáp án Bài 6

Song song với `pre_max` ta giữ `pre_arg[k]` = chỉ số `j` đạt max trong `k` phần tử đầu, và `parent[i]` = cuộc họp đứng ngay trước `i`
trong lịch tốt nhất kết thúc tại `i`. Cuối cùng lần ngược `parent` từ cuộc họp có `f` lớn nhất.

In [42]:
from bisect import bisect_right
import random

def answer_6_best_schedule(meetings):
    ms = sorted(meetings, key=lambda m: m[1])
    ends = [m[1] for m in ms]
    n = len(ms)
    pre_max = [float("-inf")] * (n + 1)
    pre_arg = [-1] * (n + 1)
    f = [0] * n
    parent = [-1] * n
    for i, (s, e, r) in enumerate(ms):
        p = bisect_right(ends, s)
        if p == 0:
            f[i] = r
        else:
            f[i] = r + s + pre_max[p]
            parent[i] = pre_arg[p]
        if f[i] - e > pre_max[i]:
            pre_max[i + 1], pre_arg[i + 1] = f[i] - e, i
        else:
            pre_max[i + 1], pre_arg[i + 1] = pre_max[i], pre_arg[i]
    last = max(range(n), key=lambda k: f[k])
    best = f[last]
    schedule = []
    while last != -1:
        schedule.append(ms[last])
        last = parent[last]
    return best, schedule[::-1]

best, schedule = answer_6_best_schedule(TRACE_MEETINGS)
print(best, schedule)
assert best == 13 and earnings_of(schedule) == 13

rng = random.Random(6)
for _ in range(200):
    m = random_small_case(rng)
    best, schedule = answer_6_best_schedule(m)
    assert best == max_earnings_bruteforce(m) == earnings_of(schedule)
    assert not any(overlaps(a, b) for a, b in combinations(schedule, 2))
print("Bài 6: PASS (lịch dựng lại hợp lệ và đạt đúng đáp án)")

13 [[0, 2, 3], [3, 5, 4], [9, 10, 1]]
Bài 6: PASS (lịch dựng lại hợp lệ và đạt đúng đáp án)


# Tóm tắt — điều cần nhớ

## Ba bước tư duy

1. **Đặt trạng thái theo "cuộc họp cuối"**: `f[i]` = thu nhập tốt nhất của lịch kết thúc tại `i`; đáp án = `max f`.
2. **Tách biến**: `f[j] + r_i + (s_i − e_j) = (r_i + s_i) + (f[j] − e_j)` → chỉ cần `max(f[j] − e_j)` trên các `j` có `e_j <= s_i`.
3. **Tiền tố + bisect**: sắp theo `end`, các `j` hợp lệ là tiền tố dài `bisect_right(ends, s_i)` → tra `pre_max` trong `O(1)`.

Câu nhớ nhanh: **"Trừ `end` khi cất vào, cộng `start` khi lấy ra."**

## Template ngắn

```python
from bisect import bisect_right

class Solution:
    def maxEarnings(self, meetings: List[List[int]]) -> int:
        ms = sorted(meetings, key=lambda m: m[1])          # sắp theo end
        ends = [m[1] for m in ms]
        pre_max = [float("-inf")] * (len(ms) + 1)          # max(f[j] - end_j), j < k
        ans = 0
        for i, (s, e, r) in enumerate(ms):
            p = bisect_right(ends, s)                      # end <= s  (nửa mở)
            f = r if p == 0 else r + s + pre_max[p]        # một mình / nối sau j tốt nhất
            ans = max(ans, f)
            pre_max[i + 1] = max(pre_max[i], f - e)        # cất vào: trừ end
        return ans
```

Độ phức tạp: **Time `O(n log n)`, Space `O(n)`**.

# Checklist trước khi submit

- [ ] Đã sắp theo **`end`** (không phải `start`) trước khi bisect trên `ends`?
- [ ] Dùng **`bisect_right`** vì khoảng **nửa mở** (`end_j == start_i` vẫn hợp lệ)?
- [ ] `pre_max[0]` là **`-inf`**, và trường hợp `p == 0` cho `f = revenue` (không cộng tiền rảnh trước cuộc họp đầu tiên)?
- [ ] Cập nhật `ans` cho **mọi** `f`, kể cả cuộc họp đơn lẻ (đáp án luôn `>= max revenue`)?
- [ ] Cập nhật `pre_max[i + 1]` bằng `f − end` (không phải `f`, không phải `f − start`)?
- [ ] Hai cuộc họp cùng `end` không bị nối với nhau (bisect_right trên `start_i < end_i` tự lo việc này)?
- [ ] Nếu viết bằng C++/Java: dùng `long long` / `long` cho `f`, `pre_max`, `ans`?
- [ ] Đã chạy lại 3 ví dụ của đề (8, 12, 11) và vài edge case: `n = 1`, chạm mút, tất cả chồng nhau, cuộc họp giữa "lỗ"?